# 🧮 Guía de IA Generativa - Embeddings, Tokens y Costes

En este notebook aprenderás:
- ✅ Qué son los embeddings y por qué son cruciales para los LLMs
- ✅ Cómo generar y comparar embeddings con modelos reales
- ✅ Qué son los tokens y cómo calcularlos
- ✅ Cómo funcionan los precios de las APIs de IA
- ✅ Estrategias para optimizar costes

**⚠️ IMPORTANTE - SEGURIDAD:**
- ❌ NUNCA compartas tus API keys en este notebook
- ❌ NO subas este archivo a GitHub si contiene keys
- ✅ Usa variables de entorno o widgets seguros
- ✅ Revoca keys si se exponen accidentalmente

---
**📋 Requisitos previos:**
- Haber completado el Notebook 1 (Introducción a IA Generativa)
- Python 3.8+
- Conexión a internet (para APIs)

# 0. SETUP Y CONTEXTO

## 📚 Instalar Librerías

Instalamos todas las librerías necesarias para trabajar con embeddings, tokenización y visualización.

In [ ]:
# 📦 Instalación de dependencias
# Esto puede tardar 1-2 minutos la primera vez

import sys
import subprocess
print("🔧 Instalando paquetes necesarios...")
print("=" * 60)

# Lista de paquetes a instalar con sus descripciones
packages = [
    ("google-genai", "🌟 Cliente de Google para Gemini (GRATIS) - Incluye embeddings"),
    ("requests", "🌐 Para hacer peticiones HTTP a APIs REST"),
    ("ipywidgets", "🎨 Widgets interactivos para inputs y visualización"),
    ("numpy", "🔢 Operaciones numéricas y vectoriales"),
    ("scikit-learn", "📊 Para PCA, métricas y visualización"),
    ("matplotlib", "📈 Gráficos y visualizaciones"),
    ("tiktoken", "🎫 Tokenizador de OpenAI (gratuito, offline)"),
    ("sentence-transformers", "🤗 Modelos de embeddings open source (HuggingFace)"),
]

for package, descripcion in packages:
    try:
        print(f"📥 Instalando {package}...")
        print(f"   ℹ️  {descripcion}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} instalado correctamente")
    except Exception as e:
        print(f"⚠️ Error instalando {package}: {e}")

print("=" * 60)
print("✨ ¡Instalación completada! Ejecuta la siguiente celda.")

## 📚 Importar Librerías

In [ ]:
# Importaciones estándar
import os
import sys
import json
import time
from datetime import datetime
from typing import List, Dict, Optional

# Librerías numéricas y científicas
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# Visualización
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
import ipywidgets as widgets
from ipywidgets import Layout, Button, Box, VBox, HBox, Text, Textarea, Password

# Tokenización (offline, gratuito)
try:
    import tiktoken
    TIKTOKEN_AVAILABLE = True
except ImportError:
    TIKTOKEN_AVAILABLE = False
    print("⚠️ tiktoken no disponible")

# Embeddings locales (HuggingFace - GRATIS)
try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMERS_AVAILABLE = True
except ImportError:
    SENTENCE_TRANSFORMERS_AVAILABLE = False
    print("⚠️ sentence-transformers no disponible")

# Google Gemini (GRATIS)
try:
    from google import genai
    GEMINI_AVAILABLE = True
except ImportError:
    GEMINI_AVAILABLE = False
    print("⚠️ Google Gemini no disponible")

# Librerías para APIs
import requests

# --------------------------------------------------
# ⚙️ CONFIGURACIÓN SSL PARA PROXIES
# --------------------------------------------------
VERIFICAR_SSL = False  # Cambia a True si NO estás detrás de un proxy

if not VERIFICAR_SSL:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print("⚠️ ADVERTENCIA: Verificación SSL desactivada")
    print("   Solo para uso en entornos con proxy")

print("\n" + "=" * 60)
print("✅ Todas las librerías importadas correctamente")
print(f"📊 Python version: {sys.version.split()[0]}")
print(f"🕐 Fecha actual: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔒 Verificación SSL: {'Activada ✅' if VERIFICAR_SSL else 'Desactivada ⚠️'}")
print(f"🤗 Sentence Transformers: {'Disponible ✅' if SENTENCE_TRANSFORMERS_AVAILABLE else 'No disponible ⚠️'}")
print(f"🎫 Tiktoken: {'Disponible ✅' if TIKTOKEN_AVAILABLE else 'No disponible ⚠️'}")

## 🔐 Configuración de API Keys (Opcional)

**💡 NOTA:** La mayoría de este notebook funciona **SIN API key** usando modelos locales.

Solo necesitarás API key si quieres probar los embeddings de Google Gemini en la sección 2.

In [ ]:
# 🔐 Widget seguro para API Key (OPCIONAL)
print("🔑 Configuración de API Key (OPCIONAL)")
print("=" * 60)
print("")
print("💡 La mayor parte de este notebook funciona SIN API key")
print("   usando modelos locales de HuggingFace.")
print("")
print("🛡️ Si introduces una key:")
print("  - Se guarda SOLO EN MEMORIA (os.environ)")
print("  - NO se guarda en el archivo .ipynb")
print("")

gemini_key_widget = Password(
    placeholder='AIza... (opcional)',
    description='Gemini:',
    disabled=False,
    layout=Layout(width='500px')
)

save_button = Button(
    description='💾 Guardar Key',
    button_style='success',
    layout=Layout(width='200px')
)

status_output = widgets.Output()

def save_keys(b):
    with status_output:
        status_output.clear_output()
        if gemini_key_widget.value and gemini_key_widget.value.startswith('AIza'):
            os.environ['GEMINI_API_KEY'] = gemini_key_widget.value
            print("✅ Key de Gemini guardada en memoria")
        else:
            print("ℹ️ Sin API key - Usaremos modelos locales gratuitos")

save_button.on_click(save_keys)

display(VBox([
    widgets.HTML("<h3>🔐 API Key de Gemini (OPCIONAL):</h3>"),
    widgets.HTML("<p style='color: #666;'>Obténla en: <a href='https://makersuite.google.com/app/apikey' target='_blank'>makersuite.google.com</a></p>"),
    gemini_key_widget,
    save_button,
    status_output
]))

print("\n💡 TIP: Puedes continuar sin key. Los ejercicios principales funcionan con modelos locales.")

## 📝 Recordatorio del Notebook 1

En el notebook anterior aprendimos:

| Concepto | Descripción |
|----------|-------------|
| **IA Generativa** | IA que crea contenido nuevo (texto, imágenes, etc.) |
| **LLM** | Large Language Model - Modelos grandes de lenguaje |
| **Prompt** | La instrucción que le damos al modelo |
| **API** | Forma programática de comunicarse con los modelos |
| **Transformers** | Arquitectura base de los modelos modernos |

### 🎯 Objetivos de este Notebook

Hoy profundizaremos en **3 conceptos clave** que apenas mencionamos:

1. **Embeddings** 🧮 - Cómo los modelos "entienden" el significado
2. **Tokens** 🎫 - La unidad de medida de los LLMs
3. **Costes** 💰 - Cómo calcular y optimizar gastos

---
# ⭐ 1. EMBEDDINGS: El idioma secreto de los LLMs

## 1.0 ¿Por qué los LLMs necesitan embeddings?

Los ordenadores no entienden palabras, solo números. Entonces... **¿cómo puede un LLM "entender" que "perro" y "can" significan lo mismo?**

### 🧠 El Problema

```
Humano ve:     "gato"  →  🐱 (concepto mental)
Ordenador ve:  "gato"  →  [103, 97, 116, 111] (códigos ASCII)
```

Con ASCII, el ordenador solo ve letras sueltas. **No sabe que "gato" se parece a "felino"**.

### 💡 La Solución: Embeddings

Un **embedding** es un vector numérico que captura el **significado** de una palabra o frase.

```
"gato"   →  [0.23, -0.45, 0.87, 0.12, ...] (768 números)
"felino" →  [0.25, -0.43, 0.85, 0.14, ...] (¡muy similar!)
"coche"  →  [-0.67, 0.91, -0.23, 0.56, ...] (muy diferente)
```

### 🎯 ¿Por qué funciona?

Los embeddings se entrenan con millones de textos. El modelo aprende que:
- Palabras que aparecen en **contextos similares** → vectores similares
- "El **gato** duerme en el sofá" / "El **felino** duerme en el sofá"

### 📊 Analogía Visual

Imagina que cada palabra es un punto en un mapa:

```
        Norte ↑ (animales)
               |
    🐱 gato    |     🐕 perro
        felino |          can
               |
 ←─────────────┼─────────────→
  Oeste        |        Este
 (pequeño)     |      (grande)
               |
        🚗 coche
          auto |
               ↓ Sur (vehículos)
```

**Palabras con significado similar** están **cerca** en el espacio vectorial.

## 1.1 Tu primer embedding (demo interactiva con modelo local)

Vamos a generar embeddings reales usando un modelo **gratuito y local** de HuggingFace.

Este modelo (`all-MiniLM-L6-v2`) es pequeño pero muy efectivo:
- ✅ **100% gratuito** - Se ejecuta en tu ordenador
- ✅ **Sin API key** - No necesita conexión tras la primera descarga
- ✅ **Rápido** - Genera embeddings en milisegundos
- ✅ **Multilingüe** - Funciona con español e inglés

In [ ]:
# 🤗 Cargar modelo de embeddings local (GRATIS)
# La primera vez descarga ~80MB, luego es instantáneo

print("🔄 Cargando modelo de embeddings local...")
print("   (Primera vez: descarga ~80MB, luego es instantáneo)")
print("")

if SENTENCE_TRANSFORMERS_AVAILABLE:
    # Usar modelo multilingüe para mejor soporte español
    modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    print("✅ Modelo cargado: paraphrase-multilingual-MiniLM-L12-v2")
    print(f"📐 Dimensión de embeddings: {modelo_embeddings.get_sentence_embedding_dimension()}")
else:
    print("❌ sentence-transformers no disponible")
    print("   Ejecuta: pip install sentence-transformers")

In [ ]:
# 🎯 Tu primer embedding

# Vamos a convertir una palabra en un vector numérico
palabra = "gato"

# Generar el embedding
embedding_gato = modelo_embeddings.encode(palabra)

print(f"🐱 Palabra: '{palabra}'")
print("")
print("📊 Su embedding (primeros 20 valores de 384):")
print(f"   {embedding_gato[:20].round(3)}")
print("")
print(f"📐 Dimensión total: {len(embedding_gato)} números")
print(f"📈 Valor mínimo: {embedding_gato.min():.4f}")
print(f"📈 Valor máximo: {embedding_gato.max():.4f}")
print(f"📈 Valor medio: {embedding_gato.mean():.4f}")

In [ ]:
# 🔍 Comparemos embeddings de palabras similares y diferentes

palabras = ["gato", "felino", "perro", "coche", "automóvil", "programación"]

print("🔄 Generando embeddings para múltiples palabras...")
print("")

embeddings_dict = {}
for palabra in palabras:
    embedding = modelo_embeddings.encode(palabra)
    embeddings_dict[palabra] = embedding
    print(f"✅ '{palabra}' → vector de {len(embedding)} dimensiones")

print("")
print("💡 Ahora tenemos cada palabra representada como un vector de 384 números.")
print("   Podemos comparar palabras midiendo la distancia entre sus vectores.")

## 1.2 Analogía visual: mapa semántico 2D

Los embeddings tienen muchas dimensiones (384 en nuestro modelo), pero podemos **proyectarlos a 2D** para visualizarlos.

Usamos una técnica llamada **PCA** (Principal Component Analysis) para reducir dimensiones.

In [ ]:
# 🗺️ Visualización: Mapa semántico 2D

# Añadimos más palabras para ver patrones interesantes
palabras_viz = [
    # Animales
    "gato", "perro", "pájaro", "pez", "caballo",
    # Vehículos
    "coche", "moto", "avión", "barco", "tren",
    # Tecnología
    "ordenador", "teléfono", "internet", "software",
    # Comida
    "pizza", "hamburguesa", "ensalada", "fruta"
]

# Generar embeddings
embeddings_viz = modelo_embeddings.encode(palabras_viz)

# Reducir a 2D con PCA
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings_viz)

# Crear gráfico
plt.figure(figsize=(12, 8))

# Colores por categoría
colores = ['red']*5 + ['blue']*5 + ['green']*4 + ['orange']*4
categorias = ['Animales']*5 + ['Vehículos']*5 + ['Tecnología']*4 + ['Comida']*4

# Scatter plot
for i, (palabra, pos) in enumerate(zip(palabras_viz, embeddings_2d)):
    plt.scatter(pos[0], pos[1], c=colores[i], s=100, alpha=0.7)
    plt.annotate(palabra, (pos[0], pos[1]), fontsize=10, ha='center', va='bottom')

# Leyenda manual
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', label='🐾 Animales'),
    Patch(facecolor='blue', label='🚗 Vehículos'),
    Patch(facecolor='green', label='💻 Tecnología'),
    Patch(facecolor='orange', label='🍕 Comida')
]
plt.legend(handles=legend_elements, loc='upper right')

plt.title('🗺️ Mapa Semántico: Palabras en Espacio 2D', fontsize=14)
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 OBSERVA: Las palabras de la misma categoría tienden a estar juntas.")
print("   Esto es porque los embeddings capturan el SIGNIFICADO, no solo las letras.")

## 1.3 Ejercicio: predice similitudes

Antes de calcular las similitudes reales, **intenta predecir** cuáles de estos pares serán más similares:

| Par | Tu predicción (1-10) |
|-----|---------------------|
| "rey" - "reina" | ¿? |
| "rey" - "presidente" | ¿? |
| "rey" - "pizza" | ¿? |
| "Madrid" - "España" | ¿? |
| "Python" - "programación" | ¿? |

In [ ]:
# 🎯 Ejercicio: Verifica tus predicciones

# Calculamos la similitud coseno (0 = nada similar, 1 = idéntico)

pares = [
    ("rey", "reina"),
    ("rey", "presidente"),
    ("rey", "pizza"),
    ("Madrid", "España"),
    ("Python", "programación")
]

print("📊 RESULTADOS - Similitud Coseno (0-1):")
print("=" * 50)

for palabra1, palabra2 in pares:
    emb1 = modelo_embeddings.encode(palabra1)
    emb2 = modelo_embeddings.encode(palabra2)
    
    # Similitud coseno
    similitud = cosine_similarity([emb1], [emb2])[0][0]
    
    # Barra visual
    barra = "█" * int(similitud * 20)
    
    print(f"'{palabra1}' ↔ '{palabra2}'")
    print(f"   Similitud: {similitud:.3f} {barra}")
    print()

print("💡 ¿Coincidieron con tus predicciones?")

---
# 🔬 2. EMBEDDINGS EN ACCIÓN

## 2.1 Generador de embeddings interactivo

Prueba a generar embeddings de cualquier texto que quieras.

In [ ]:
# 🎮 Generador interactivo de embeddings

# Widget de entrada
texto_input = Textarea(
    value='Escribe aquí tu texto para generar su embedding',
    placeholder='Cualquier palabra o frase...',
    description='Texto:',
    layout=Layout(width='600px', height='80px')
)

generar_btn = Button(
    description='🔄 Generar Embedding',
    button_style='primary',
    layout=Layout(width='200px')
)

output_embedding = widgets.Output()

def generar_embedding(b):
    with output_embedding:
        output_embedding.clear_output()
        texto = texto_input.value.strip()
        
        if not texto:
            print("⚠️ Escribe algo de texto primero")
            return
        
        # Generar embedding
        embedding = modelo_embeddings.encode(texto)
        
        print(f"📝 Texto: '{texto}'")
        print(f"\n📊 Embedding generado:")
        print(f"   Dimensiones: {len(embedding)}")
        print(f"   Primeros 10 valores: {embedding[:10].round(4)}")
        print(f"\n📈 Estadísticas:")
        print(f"   Mínimo: {embedding.min():.4f}")
        print(f"   Máximo: {embedding.max():.4f}")
        print(f"   Media: {embedding.mean():.4f}")
        print(f"   Norma L2: {np.linalg.norm(embedding):.4f}")

generar_btn.on_click(generar_embedding)

display(VBox([texto_input, generar_btn, output_embedding]))

## 2.2 Calculadora de similitud coseno

La **similitud coseno** mide el ángulo entre dos vectores:
- **1.0** = Vectores apuntan en la misma dirección (muy similares)
- **0.0** = Vectores perpendiculares (sin relación)
- **-1.0** = Vectores opuestos (significados opuestos)

```
                    ↑ B
                   /
                  / θ (ángulo pequeño = similar)
                 /
    ────────────→ A
    
    similitud_coseno = cos(θ)
```

In [ ]:
# 🔢 Calculadora de similitud coseno interactiva

texto1_input = Text(
    value='gato',
    description='Texto 1:',
    layout=Layout(width='400px')
)

texto2_input = Text(
    value='perro',
    description='Texto 2:',
    layout=Layout(width='400px')
)

comparar_btn = Button(
    description='📊 Calcular Similitud',
    button_style='success',
    layout=Layout(width='200px')
)

output_similitud = widgets.Output()

def calcular_similitud(b):
    with output_similitud:
        output_similitud.clear_output()
        
        t1 = texto1_input.value.strip()
        t2 = texto2_input.value.strip()
        
        if not t1 or not t2:
            print("⚠️ Escribe ambos textos")
            return
        
        # Generar embeddings
        emb1 = modelo_embeddings.encode(t1)
        emb2 = modelo_embeddings.encode(t2)
        
        # Calcular similitud
        sim = cosine_similarity([emb1], [emb2])[0][0]
        
        # Visualización
        barra = "█" * int(sim * 30)
        espacios = "░" * (30 - int(sim * 30))
        
        print(f"📝 Texto 1: '{t1}'")
        print(f"📝 Texto 2: '{t2}'")
        print(f"\n📊 Similitud Coseno: {sim:.4f}")
        print(f"\n   0.0 [{barra}{espacios}] 1.0")
        print()
        
        # Interpretación
        if sim > 0.8:
            print("🟢 MUY SIMILAR - Significados casi idénticos")
        elif sim > 0.5:
            print("🟡 RELACIONADOS - Comparten contexto semántico")
        elif sim > 0.3:
            print("🟠 ALGO RELACIONADOS - Conexión débil")
        else:
            print("🔴 POCO RELACIONADOS - Significados diferentes")

comparar_btn.on_click(calcular_similitud)

display(VBox([texto1_input, texto2_input, comparar_btn, output_similitud]))

## 2.3 Visualización: Matriz de similitudes

Veamos cómo se relacionan múltiples palabras entre sí con un **heatmap**.

In [ ]:
# 🎨 Matriz de similitudes (Heatmap)

palabras_matriz = [
    "perro", "gato", "mascota",  # Animales/mascotas
    "coche", "moto", "vehículo",  # Transporte
    "Python", "código", "programar",  # Programación
    "pizza", "comida", "restaurante"  # Comida
]

# Generar embeddings
embeddings_matriz = modelo_embeddings.encode(palabras_matriz)

# Calcular matriz de similitud
matriz_sim = cosine_similarity(embeddings_matriz)

# Visualizar
plt.figure(figsize=(10, 8))
plt.imshow(matriz_sim, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(label='Similitud Coseno')

# Etiquetas
plt.xticks(range(len(palabras_matriz)), palabras_matriz, rotation=45, ha='right')
plt.yticks(range(len(palabras_matriz)), palabras_matriz)

# Añadir valores en las celdas
for i in range(len(palabras_matriz)):
    for j in range(len(palabras_matriz)):
        color = 'white' if matriz_sim[i,j] > 0.5 else 'black'
        plt.text(j, i, f'{matriz_sim[i,j]:.2f}', ha='center', va='center', 
                 color=color, fontsize=8)

plt.title('🎨 Matriz de Similitud Semántica', fontsize=14)
plt.tight_layout()
plt.show()

print("\n💡 OBSERVA:")
print("   - La diagonal siempre es 1.0 (palabra consigo misma)")
print("   - Los bloques verdes muestran palabras relacionadas")
print("   - 'mascota' tiene alta similitud con 'perro' y 'gato'")

## 2.4 Mini buscador semántico

Una aplicación práctica de embeddings: un **buscador que entiende el significado**.

A diferencia de buscar palabras exactas (como Ctrl+F), este buscador encuentra **documentos semánticamente similares** a tu consulta.

In [ ]:
# 🔍 Mini Buscador Semántico

# Base de datos de documentos de ejemplo
documentos = [
    "El gato duerme tranquilamente en el sofá del salón",
    "Mi perro ladra cada vez que pasa el cartero por la puerta",
    "Python es un lenguaje de programación muy popular para data science",
    "El coche eléctrico contamina menos que los vehículos de gasolina",
    "La pizza italiana lleva tomate, mozzarella y albahaca fresca",
    "El machine learning permite a las máquinas aprender de datos",
    "Los pájaros migran hacia el sur cuando llega el invierno",
    "JavaScript se usa principalmente para desarrollo web frontend",
    "El tren de alta velocidad conecta Madrid con Barcelona en 2 horas",
    "La inteligencia artificial está transformando muchas industrias"
]

# Pre-calcular embeddings de todos los documentos
print("🔄 Indexando documentos...")
embeddings_docs = modelo_embeddings.encode(documentos)
print(f"✅ {len(documentos)} documentos indexados\n")

# Función de búsqueda
def buscar(consulta, top_k=3):
    # Embedding de la consulta
    emb_consulta = modelo_embeddings.encode(consulta)
    
    # Calcular similitudes
    similitudes = cosine_similarity([emb_consulta], embeddings_docs)[0]
    
    # Ordenar por similitud
    indices_ordenados = similitudes.argsort()[::-1][:top_k]
    
    print(f"🔍 Búsqueda: '{consulta}'")
    print("=" * 60)
    print()
    
    for i, idx in enumerate(indices_ordenados, 1):
        sim = similitudes[idx]
        doc = documentos[idx]
        barra = "█" * int(sim * 20)
        print(f"#{i} Similitud: {sim:.3f} {barra}")
        print(f"   📄 {doc}")
        print()

# Ejemplos de búsqueda
buscar("animales domésticos")

In [ ]:
# Prueba con otras consultas
buscar("desarrollo de software")

In [ ]:
# 🎮 Buscador interactivo

busqueda_input = Text(
    value='',
    placeholder='Escribe tu búsqueda...',
    description='Buscar:',
    layout=Layout(width='500px')
)

buscar_btn = Button(
    description='🔍 Buscar',
    button_style='primary',
    layout=Layout(width='150px')
)

output_busqueda = widgets.Output()

def ejecutar_busqueda(b):
    with output_busqueda:
        output_busqueda.clear_output()
        consulta = busqueda_input.value.strip()
        if consulta:
            buscar(consulta)
        else:
            print("⚠️ Escribe algo para buscar")

buscar_btn.on_click(ejecutar_busqueda)

print("📚 Base de datos de documentos:")
for i, doc in enumerate(documentos, 1):
    print(f"   {i}. {doc[:50]}...")
print()

display(VBox([busqueda_input, buscar_btn, output_busqueda]))

---
# 🏪 3. MODELOS DE EMBEDDINGS

## 3.1 Proveedores principales

Existen múltiples proveedores de modelos de embeddings. Cada uno tiene sus ventajas:

### 🔷 OpenAI
- **Modelo**: `text-embedding-3-small`, `text-embedding-3-large`
- **Dimensiones**: 1536 / 3072
- **Precio**: $0.00002 / 1K tokens (small)
- **Ventajas**: Alta calidad, API sencilla

### 🔷 Google (Gemini)
- **Modelo**: `text-embedding-004`
- **Dimensiones**: 768
- **Precio**: **GRATIS** hasta límite
- **Ventajas**: Gratis, multilingüe

### 🔷 Cohere
- **Modelo**: `embed-multilingual-v3.0`
- **Dimensiones**: 1024
- **Precio**: $0.0001 / 1K tokens
- **Ventajas**: Excelente multilingüe

### 🔷 Voyage AI
- **Modelo**: `voyage-large-2`
- **Dimensiones**: 1536
- **Precio**: $0.00012 / 1K tokens
- **Ventajas**: Máxima calidad en retrieval

## 3.2 Modelos Open Source (GRATIS)

Hay excelentes opciones **100% gratuitas** que puedes ejecutar localmente:

### 🤗 HuggingFace / Sentence Transformers

| Modelo | Dimensiones | Idiomas | Tamaño | Uso ideal |
|--------|-------------|---------|--------|------------|
| `all-MiniLM-L6-v2` | 384 | EN | 80MB | Rápido, general |
| `all-mpnet-base-v2` | 768 | EN | 420MB | Alta calidad EN |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | 50+ | 470MB | **Multilingüe** |
| `multilingual-e5-large` | 1024 | 100+ | 2.2GB | Máxima calidad multilingüe |

### 🦙 Ollama (Local)
- `nomic-embed-text` - 768 dim, muy bueno
- `mxbai-embed-large` - 1024 dim, alta calidad

## 3.3 Tabla comparativa completa

| Modelo | Proveedor | Dim. | Idiomas | Coste/1M tokens | Calidad MTEB |
|--------|-----------|------|---------|-----------------|-------------|
| `text-embedding-3-large` | OpenAI | 3072 | Multi | $0.13 | ⭐⭐⭐⭐⭐ |
| `text-embedding-3-small` | OpenAI | 1536 | Multi | $0.02 | ⭐⭐⭐⭐ |
| `text-embedding-004` | Google | 768 | Multi | **GRATIS** | ⭐⭐⭐⭐ |
| `embed-multilingual-v3` | Cohere | 1024 | 100+ | $0.10 | ⭐⭐⭐⭐⭐ |
| `voyage-large-2` | Voyage | 1536 | Multi | $0.12 | ⭐⭐⭐⭐⭐ |
| `multilingual-e5-large` | HuggingFace | 1024 | 100+ | **GRATIS** | ⭐⭐⭐⭐ |
| `all-MiniLM-L6-v2` | HuggingFace | 384 | EN | **GRATIS** | ⭐⭐⭐ |

*MTEB = Massive Text Embedding Benchmark*

## 3.4 Ejercicio: ¿Qué modelo para cada caso?

Basándote en la tabla anterior, elige el mejor modelo para cada escenario:

| Escenario | Tu elección | Por qué |
|-----------|-------------|----------|
| Startup sin presupuesto, solo español | | |
| Empresa con budget, máxima calidad | | |
| Aplicación offline, sin internet | | |
| Prototipo rápido, cualquier idioma | | |

In [ ]:
# 💡 Respuestas sugeridas (ejecuta para ver)

respuestas = """
📋 RESPUESTAS SUGERIDAS:

1. Startup sin presupuesto, solo español:
   → paraphrase-multilingual-MiniLM-L12-v2 (HuggingFace)
   ✅ Gratis, buen español, ligero

2. Empresa con budget, máxima calidad:
   → text-embedding-3-large (OpenAI) o voyage-large-2
   ✅ Máxima calidad en benchmarks

3. Aplicación offline, sin internet:
   → multilingual-e5-large (HuggingFace) o nomic-embed (Ollama)
   ✅ Se ejecuta 100% local

4. Prototipo rápido, cualquier idioma:
   → text-embedding-004 (Google Gemini)
   ✅ Gratis, API sencilla, multilingüe
"""

print(respuestas)

---
# 🎫 4. TOKENS: La moneda del mundo LLM

## 4.1 ¿Qué es un token?

Un **token** NO es lo mismo que una palabra. Es la unidad mínima que procesa un LLM.

### 🔍 Ejemplos de tokenización:

```
"Hola"           → ["Hola"]                    (1 token)
"inteligencia"   → ["int", "elig", "encia"]    (3 tokens)
"ChatGPT"        → ["Chat", "G", "PT"]         (3 tokens)
"🚀"             → ["\xf0\x9f\x9a\x80"]        (1-4 tokens)
```

### 🧩 ¿Por qué tokens y no palabras?

1. **Vocabulario limitado**: Un modelo no puede memorizar todas las palabras posibles
2. **Subpalabras**: Permite manejar palabras nuevas/raras descomponiéndolas
3. **Multilingüe**: Funciona para cualquier idioma
4. **Eficiencia**: Balance entre vocabulario y longitud de secuencia

### 📊 Métodos de tokenización populares:

| Método | Usado por | Características |
|--------|-----------|------------------|
| **BPE** (Byte Pair Encoding) | GPT, Llama | Aprende pares frecuentes |
| **WordPiece** | BERT, Gemini | Similar a BPE, usado por Google |
| **SentencePiece** | T5, Llama | Independiente del idioma |

## 4.2 Tipos de tokens

Cuando usas un LLM, hay **diferentes tipos** de tokens que afectan al coste:

### 📥 Input tokens (Prompt)
- Lo que TÚ envías al modelo
- Incluye: instrucciones, contexto, preguntas
- Generalmente más baratos

### 📤 Output tokens (Completion)
- Lo que el modelo GENERA
- La respuesta que recibes
- Generalmente más caros (2-4x)

### 🧠 Reasoning tokens (Nuevo en 2024-2025)
- Tokens de "pensamiento interno" (o1, DeepSeek R1)
- No siempre visibles al usuario
- Se cobran aparte

```
┌────────────────────────────────────────┐
│           PETICIÓN A UN LLM            │
├────────────────────────────────────────┤
│                                        │
│  📥 INPUT (tu prompt):                 │
│  "Explica qué es Python"              │
│  → ~5 tokens × $0.001/1K = $0.000005  │
│                                        │
│  🧠 REASONING (interno):               │
│  [pensamiento oculto del modelo...]    │
│  → ~50 tokens × $0.002/1K = $0.0001   │
│                                        │
│  📤 OUTPUT (respuesta):                │
│  "Python es un lenguaje de..."        │
│  → ~100 tokens × $0.003/1K = $0.0003  │
│                                        │
│  💰 COSTE TOTAL: ~$0.0004              │
└────────────────────────────────────────┘
```

## 4.3 Contador de tokens interactivo

Vamos a ver cómo se tokeniza texto real usando **tiktoken** (tokenizador de OpenAI, gratuito y offline).

In [ ]:
# 🎫 Cargar tokenizador

if TIKTOKEN_AVAILABLE:
    # Usamos el tokenizador de GPT-4 (cl100k_base)
    tokenizer = tiktoken.get_encoding("cl100k_base")
    print("✅ Tokenizador cargado: cl100k_base (usado por GPT-3.5/4)")
else:
    print("❌ tiktoken no disponible. Instala con: pip install tiktoken")

In [ ]:
# 📊 Visualizar tokenización de un texto

def visualizar_tokens(texto):
    """Muestra cómo se tokeniza un texto"""
    tokens = tokenizer.encode(texto)
    
    print(f"📝 Texto: '{texto}'")
    print(f"📊 Número de tokens: {len(tokens)}")
    print(f"📐 Caracteres: {len(texto)}")
    print(f"📈 Ratio tokens/caracteres: {len(tokens)/len(texto):.3f}")
    print()
    print("🔍 Tokens individuales:")
    
    for i, token_id in enumerate(tokens):
        token_text = tokenizer.decode([token_id])
        # Mostrar espacios de forma visible
        display_text = token_text.replace(' ', '␣')
        print(f"   [{i+1}] ID={token_id:6d} → '{display_text}'")

# Ejemplos
visualizar_tokens("Hola mundo")

In [ ]:
# 🔍 Comparar tokenización de diferentes textos

textos_ejemplo = [
    "Hello world",
    "Hola mundo",
    "inteligencia artificial",
    "artificial intelligence",
    "ChatGPT es genial",
    "🚀🎉🔥",
    "12345",
    "supercalifragilistico"
]

print("📊 COMPARATIVA DE TOKENIZACIÓN")
print("=" * 70)
print(f"{'Texto':<35} {'Chars':>8} {'Tokens':>8} {'Ratio':>8}")
print("-" * 70)

for texto in textos_ejemplo:
    tokens = tokenizer.encode(texto)
    ratio = len(tokens) / len(texto) if len(texto) > 0 else 0
    print(f"{texto:<35} {len(texto):>8} {len(tokens):>8} {ratio:>8.3f}")

print()
print("💡 OBSERVA:")
print("   - Palabras comunes en inglés = menos tokens")
print("   - Palabras largas se dividen en subpalabras")
print("   - Emojis consumen varios tokens")
print("   - Números son relativamente eficientes")

In [ ]:
# 🎮 Contador interactivo de tokens

texto_token_input = Textarea(
    value='Escribe aquí para contar tokens...',
    description='Texto:',
    layout=Layout(width='600px', height='100px')
)

contar_btn = Button(
    description='🔢 Contar Tokens',
    button_style='info',
    layout=Layout(width='200px')
)

output_tokens = widgets.Output()

def contar_tokens(b):
    with output_tokens:
        output_tokens.clear_output()
        texto = texto_token_input.value
        
        if not texto.strip():
            print("⚠️ Escribe algo de texto")
            return
        
        tokens = tokenizer.encode(texto)
        palabras = len(texto.split())
        caracteres = len(texto)
        
        print("📊 ANÁLISIS DE TOKENS")
        print("=" * 40)
        print(f"📝 Palabras: {palabras}")
        print(f"📐 Caracteres: {caracteres}")
        print(f"🎫 Tokens: {len(tokens)}")
        print()
        print(f"📈 Ratio tokens/palabra: {len(tokens)/palabras:.2f}" if palabras > 0 else "")
        print()
        print("💰 COSTE ESTIMADO (GPT-4):")
        coste_input = (len(tokens) / 1000) * 0.03  # $0.03/1K input
        print(f"   Como INPUT: ${coste_input:.6f}")
        coste_output = (len(tokens) / 1000) * 0.06  # $0.06/1K output
        print(f"   Como OUTPUT: ${coste_output:.6f}")

contar_btn.on_click(contar_tokens)

display(VBox([texto_token_input, contar_btn, output_tokens]))

## 4.4 Regla práctica: 1 token ≈ 0.75 palabras

Para **estimaciones rápidas**, puedes usar esta regla:

```
┌────────────────────────────────────────┐
│     REGLAS DE APROXIMACIÓN             │
├────────────────────────────────────────┤
│                                        │
│  📝 INGLÉS:                            │
│     1 token ≈ 4 caracteres            │
│     1 token ≈ 0.75 palabras           │
│     100 palabras ≈ 133 tokens         │
│                                        │
│  📝 ESPAÑOL:                           │
│     1 token ≈ 3-3.5 caracteres        │
│     1 token ≈ 0.6-0.7 palabras        │
│     100 palabras ≈ 140-150 tokens     │
│                                        │
│  📖 REFERENCIAS:                       │
│     1 página A4 ≈ 500 palabras        │
│                  ≈ 650-750 tokens     │
│     1 libro (300 pág) ≈ 200K tokens   │
│                                        │
└────────────────────────────────────────┘
```

### ⚠️ Importante:
- El español suele necesitar **más tokens** que el inglés para el mismo contenido
- Código fuente puede ser muy variable
- Siempre verifica con el contador real para estimaciones de coste

---
# 💰 5. PRECIOS Y CALCULADORA DE COSTES

## 5.1 Cómo leen los precios los proveedores

Los precios de APIs de LLM se expresan normalmente en **$/1K tokens** o **$/1M tokens**.

### 📖 Cómo leer una tabla de precios:

```
┌─────────────────────────────────────────────────────────┐
│  GPT-4 Turbo                                            │
│  ──────────────────────────────────────────────         │
│  Input:  $0.01/1K tokens = $10/1M tokens                │
│  Output: $0.03/1K tokens = $30/1M tokens                │
│                                                         │
│  Ejemplo: Prompt de 500 tokens + Respuesta de 200       │
│           = (500 × $0.01/1000) + (200 × $0.03/1000)     │
│           = $0.005 + $0.006 = $0.011                    │
└─────────────────────────────────────────────────────────┘
```

### 📊 Tabla de precios (Marzo 2026):

| Modelo | Input ($/1M) | Output ($/1M) | Contexto |
|--------|--------------|---------------|----------|
| **GPT-4o** | $2.50 | $10.00 | 128K |
| **GPT-4o-mini** | $0.15 | $0.60 | 128K |
| **Claude 3.5 Sonnet** | $3.00 | $15.00 | 200K |
| **Claude 3 Haiku** | $0.25 | $1.25 | 200K |
| **Gemini 1.5 Pro** | $1.25 | $5.00 | 1M |
| **Gemini 1.5 Flash** | **GRATIS** | **GRATIS** | 1M |

## 5.2 Ejercicio: Cálculo manual de costes

**Escenario**: Tienes una app que procesa emails de clientes.

- Email promedio: 200 palabras ≈ 270 tokens (input)
- Respuesta generada: 100 palabras ≈ 135 tokens (output)
- Volumen: 1,000 emails/día

**Calcula el coste diario con GPT-4o y con GPT-4o-mini:**

In [ ]:
# 💡 Solución del ejercicio

# Datos del escenario
tokens_input_por_email = 270
tokens_output_por_email = 135
emails_por_dia = 1000

# Totales diarios
total_input = tokens_input_por_email * emails_por_dia
total_output = tokens_output_por_email * emails_por_dia

print("📊 CÁLCULO DE COSTES DIARIOS")
print("=" * 50)
print(f"📧 Emails/día: {emails_por_dia:,}")
print(f"📥 Tokens input/día: {total_input:,}")
print(f"📤 Tokens output/día: {total_output:,}")
print()

# GPT-4o
coste_gpt4o_input = (total_input / 1_000_000) * 2.50
coste_gpt4o_output = (total_output / 1_000_000) * 10.00
coste_gpt4o_total = coste_gpt4o_input + coste_gpt4o_output

print("💰 GPT-4o ($2.50/$10.00 por 1M):")
print(f"   Input:  ${coste_gpt4o_input:.4f}")
print(f"   Output: ${coste_gpt4o_output:.4f}")
print(f"   TOTAL:  ${coste_gpt4o_total:.4f}/día")
print(f"   MENSUAL: ${coste_gpt4o_total * 30:.2f}")
print()

# GPT-4o-mini
coste_mini_input = (total_input / 1_000_000) * 0.15
coste_mini_output = (total_output / 1_000_000) * 0.60
coste_mini_total = coste_mini_input + coste_mini_output

print("💰 GPT-4o-mini ($0.15/$0.60 por 1M):")
print(f"   Input:  ${coste_mini_input:.6f}")
print(f"   Output: ${coste_mini_output:.6f}")
print(f"   TOTAL:  ${coste_mini_total:.4f}/día")
print(f"   MENSUAL: ${coste_mini_total * 30:.2f}")
print()

ahorro = ((coste_gpt4o_total - coste_mini_total) / coste_gpt4o_total) * 100
print(f"💡 Ahorro usando mini: {ahorro:.1f}%")

## 5.3 Widget calculadora interactiva

In [ ]:
# 🧮 Calculadora de Costes Interactiva

# Modelos y precios ($/1M tokens)
modelos = {
    "GPT-4o": {"input": 2.50, "output": 10.00},
    "GPT-4o-mini": {"input": 0.15, "output": 0.60},
    "Claude 3.5 Sonnet": {"input": 3.00, "output": 15.00},
    "Claude 3 Haiku": {"input": 0.25, "output": 1.25},
    "Gemini 1.5 Pro": {"input": 1.25, "output": 5.00},
    "Gemini Flash (GRATIS)": {"input": 0.00, "output": 0.00},
}

# Widgets
modelo_dropdown = widgets.Dropdown(
    options=list(modelos.keys()),
    value="GPT-4o-mini",
    description='Modelo:',
    layout=Layout(width='300px')
)

input_tokens_slider = widgets.IntSlider(
    value=1000,
    min=0,
    max=100000,
    step=100,
    description='Input tokens:',
    layout=Layout(width='400px')
)

output_tokens_slider = widgets.IntSlider(
    value=500,
    min=0,
    max=50000,
    step=100,
    description='Output tokens:',
    layout=Layout(width='400px')
)

peticiones_slider = widgets.IntSlider(
    value=100,
    min=1,
    max=10000,
    step=10,
    description='Peticiones:',
    layout=Layout(width='400px')
)

output_calculadora = widgets.Output()

def calcular_coste(change):
    with output_calculadora:
        output_calculadora.clear_output()
        
        modelo = modelo_dropdown.value
        precios = modelos[modelo]
        
        input_tok = input_tokens_slider.value
        output_tok = output_tokens_slider.value
        peticiones = peticiones_slider.value
        
        # Calcular
        total_input = input_tok * peticiones
        total_output = output_tok * peticiones
        
        coste_input = (total_input / 1_000_000) * precios["input"]
        coste_output = (total_output / 1_000_000) * precios["output"]
        coste_total = coste_input + coste_output
        
        print(f"💻 Modelo: {modelo}")
        print(f"📊 Precios: ${precios['input']}/1M input, ${precios['output']}/1M output")
        print("=" * 50)
        print(f"📥 Input total: {total_input:,} tokens")
        print(f"📤 Output total: {total_output:,} tokens")
        print()
        print(f"💰 Coste input:  ${coste_input:.6f}")
        print(f"💰 Coste output: ${coste_output:.6f}")
        print(f"💰 COSTE TOTAL:  ${coste_total:.6f}")
        print()
        print(f"📅 Si esto es diario:")
        print(f"   Semanal:  ${coste_total * 7:.4f}")
        print(f"   Mensual:  ${coste_total * 30:.4f}")
        print(f"   Anual:    ${coste_total * 365:.2f}")

# Conectar eventos
modelo_dropdown.observe(calcular_coste, names='value')
input_tokens_slider.observe(calcular_coste, names='value')
output_tokens_slider.observe(calcular_coste, names='value')
peticiones_slider.observe(calcular_coste, names='value')

print("🧮 CALCULADORA DE COSTES LLM")
print("Ajusta los parámetros para ver el coste estimado:\n")

display(VBox([
    modelo_dropdown,
    input_tokens_slider,
    output_tokens_slider,
    peticiones_slider,
    output_calculadora
]))

# Calcular inicial
calcular_coste(None)

## 5.4 Comparativa de precios: 5 modelos top

In [ ]:
# 📊 Comparativa visual de precios

# Datos
modelos_comp = ['GPT-4o', 'GPT-4o-mini', 'Claude 3.5\nSonnet', 'Claude 3\nHaiku', 'Gemini 1.5\nPro']
precios_input = [2.50, 0.15, 3.00, 0.25, 1.25]
precios_output = [10.00, 0.60, 15.00, 1.25, 5.00]

x = np.arange(len(modelos_comp))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, precios_input, width, label='Input ($/1M)', color='steelblue')
bars2 = ax.bar(x + width/2, precios_output, width, label='Output ($/1M)', color='coral')

# Etiquetas
ax.set_ylabel('Precio ($ por 1M tokens)')
ax.set_title('💰 Comparativa de Precios - Principales LLMs (Marzo 2026)')
ax.set_xticks(x)
ax.set_xticklabels(modelos_comp)
ax.legend()

# Añadir valores encima de las barras
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'${height}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'${height}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\n💡 OBSERVACIONES:")
print("   - Los modelos 'mini' o 'Haiku' son 10-20x más baratos")
print("   - Output SIEMPRE cuesta más que input (2-5x)")
print("   - Gemini Flash es GRATIS dentro de sus límites")

---
# 🎯 6. OPTIMIZA TUS TOKENS (AHORRA 💸)

## 6.1 Prompt largo vs. prompt corto

Menos tokens = menos coste. Pero hay un equilibrio entre brevedad y claridad.

In [ ]:
# 📊 Comparación: Mismo objetivo, diferentes longitudes de prompt

prompt_largo = """
Eres un asistente de inteligencia artificial muy útil y amigable. Tu objetivo es ayudar 
a los usuarios de la mejor manera posible. Siempre debes ser respetuoso, profesional y 
proporcionar respuestas precisas y bien estructuradas.

Ahora, necesito que me ayudes con lo siguiente: Por favor, proporcióname un resumen 
conciso pero completo del siguiente texto, asegurándote de capturar los puntos principales 
y la idea central del mismo. El resumen debe ser fácil de entender y no debe exceder 
los 100 palabras aproximadamente:

Texto a resumir: La inteligencia artificial está transformando múltiples industrias.
"""

prompt_corto = """
Resume en 2 frases: La inteligencia artificial está transformando múltiples industrias.
"""

# Calcular tokens
tokens_largo = len(tokenizer.encode(prompt_largo))
tokens_corto = len(tokenizer.encode(prompt_corto))

print("📊 COMPARACIÓN DE PROMPTS")
print("=" * 50)
print()
print("📝 PROMPT LARGO:")
print(f"   Tokens: {tokens_largo}")
print(f"   Coste (GPT-4o): ${(tokens_largo/1_000_000)*2.50:.6f}")
print()
print("📝 PROMPT CORTO:")
print(f"   Tokens: {tokens_corto}")
print(f"   Coste (GPT-4o): ${(tokens_corto/1_000_000)*2.50:.6f}")
print()
print(f"💡 AHORRO: {((tokens_largo - tokens_corto)/tokens_largo)*100:.1f}% menos tokens")
print(f"   En 10,000 peticiones: ${((tokens_largo - tokens_corto)*10000/1_000_000)*2.50:.2f} ahorrados")

## 6.2 Recortar contexto innecesario

Cuando incluyes contexto (documentos, historial, etc.), elimina lo que no sea relevante.

In [ ]:
# 🔪 Ejemplo: Recortar contexto innecesario

contexto_completo = """
Email #1 (hace 3 meses): Hola, quería preguntar sobre el horario de apertura...
Email #2 (hace 2 meses): Gracias por la información, muy útil...
Email #3 (hace 1 mes): Tengo una queja sobre un producto defectuoso...
Email #4 (hace 1 semana): Seguimiento de mi queja anterior, sigo esperando...
Email #5 (hoy): ¿Pueden darme una actualización del estado de mi reclamación?
"""

# Estrategias de recorte:

# 1. Solo últimos N emails
contexto_reciente = """
Email #4 (hace 1 semana): Seguimiento de mi queja anterior, sigo esperando...
Email #5 (hoy): ¿Pueden darme una actualización del estado de mi reclamación?
"""

# 2. Resumen del historial + último email
contexto_resumido = """
[Resumen: Cliente con queja de producto desde hace 1 mes, esperando resolución]
Email actual: ¿Pueden darme una actualización del estado de mi reclamación?
"""

tokens_completo = len(tokenizer.encode(contexto_completo))
tokens_reciente = len(tokenizer.encode(contexto_reciente))
tokens_resumido = len(tokenizer.encode(contexto_resumido))

print("📊 ESTRATEGIAS DE RECORTE DE CONTEXTO")
print("=" * 50)
print(f"\n📄 Contexto completo: {tokens_completo} tokens")
print(f"📄 Solo recientes:    {tokens_reciente} tokens ({100-tokens_reciente*100//tokens_completo}% menos)")
print(f"📄 Con resumen:       {tokens_resumido} tokens ({100-tokens_resumido*100//tokens_completo}% menos)")
print()
print("💡 CONSEJOS:")
print("   - Limita el historial a los últimos N mensajes relevantes")
print("   - Usa resúmenes para contexto antiguo")
print("   - Elimina saludos, firmas y contenido repetitivo")

## 6.3 Limitar max_tokens en output

Puedes controlar cuántos tokens genera el modelo con el parámetro `max_tokens`.

In [ ]:
# ⚙️ Impacto de max_tokens

print("💡 CÓMO USAR max_tokens")
print("=" * 50)
print("""
Cuando llamas a una API de LLM, puedes limitar la respuesta:

✅ EJEMPLO (Python con OpenAI):

```python
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explica Python"}],
    max_tokens=100  # ← Limita la respuesta
)
```

📊 IMPACTO EN COSTES:

┌─────────────────────────────────────────────┐
│  Sin límite:   ~500 tokens output = $0.005  │
│  max_tokens=100: 100 tokens output = $0.001 │
│  AHORRO: 80%                                │
└─────────────────────────────────────────────┘

⚠️ CUIDADO:
- Si el límite es muy bajo, la respuesta se cortará
- Ajusta según la tarea (resumen corto vs explicación larga)
""")

## 6.4 Ejercicio: Optimiza este prompt

El siguiente prompt es ineficiente. Tu objetivo es reescribirlo para **reducir tokens** sin perder claridad.

In [ ]:
# 🎯 EJERCICIO: Optimiza este prompt

prompt_ineficiente = """
Hola, espero que estés bien. Soy un desarrollador de software y estoy trabajando en un 
proyecto muy interesante. Me gustaría pedirte un favor si no es mucha molestia. Verás, 
necesito que me ayudes a escribir una función en el lenguaje de programación Python que 
sea capaz de realizar la siguiente tarea específica: tomar una lista de números enteros 
como parámetro de entrada y devolver solamente aquellos números que sean pares, es decir, 
que sean divisibles por 2 sin dejar resto. La función debe ser eficiente, bien documentada 
con comentarios explicativos, y seguir las mejores prácticas de programación en Python. 
Además, si pudieras incluir algunos ejemplos de uso de la función para que quede más claro 
cómo utilizarla, te lo agradecería enormemente. Muchas gracias de antemano por tu ayuda.
"""

tokens_original = len(tokenizer.encode(prompt_ineficiente))
print(f"📝 Prompt original: {tokens_original} tokens")
print()
print("🎯 TU TAREA: Reescribe este prompt reduciendo tokens al máximo")
print("   Mantén la misma funcionalidad solicitada")
print()
print("💡 PISTAS:")
print("   - Elimina saludos y despedidas")
print("   - Quita explicaciones obvias")
print("   - Sé directo y conciso")
print("   - Usa términos técnicos precisos")

In [ ]:
# ✅ SOLUCIÓN PROPUESTA

prompt_optimizado = """
Escribe una función Python que filtre números pares de una lista. 
Incluye docstring y un ejemplo de uso.
"""

tokens_optimizado = len(tokenizer.encode(prompt_optimizado))

print("📊 COMPARACIÓN:")
print("=" * 50)
print(f"❌ Original:   {tokens_original} tokens")
print(f"✅ Optimizado: {tokens_optimizado} tokens")
print()
ahorro_pct = ((tokens_original - tokens_optimizado) / tokens_original) * 100
print(f"💰 AHORRO: {ahorro_pct:.1f}% menos tokens")
print()
print("📝 Prompt optimizado:")
print(prompt_optimizado)
print()
print("💡 El modelo entiende perfectamente la tarea con mucho menos texto.")

---
# 📋 RESUMEN Y CONCLUSIONES

## 🎓 Lo que has aprendido

### ⭐ Embeddings
- Son **vectores numéricos** que representan el **significado** del texto
- Permiten **comparar** textos por similitud semántica
- Aplicaciones: búsqueda semántica, clustering, detección de duplicados
- Opciones **gratuitas**: HuggingFace, Google Gemini (con límites)

### 🎫 Tokens
- No son palabras ni caracteres, son **subunidades** de texto
- **Regla práctica**: 1 token ≈ 0.75 palabras (en inglés)
- **Tipos**: input (más barato), output (más caro), reasoning (nuevo)

### 💰 Costes
- Se cobran por **$/1M tokens** (input y output separados)
- Los modelos "mini" son **10-20x más baratos**
- **Gemini Flash es GRATIS** dentro de sus límites generosos

### 🎯 Optimización
- **Prompts concisos** = menos tokens = menos coste
- **Recorta contexto** innecesario
- Usa **max_tokens** para limitar output
- Elige el **modelo adecuado** para cada tarea

## 🔗 Recursos adicionales

- 📖 [Tokenizer interactivo de OpenAI](https://platform.openai.com/tokenizer)
- 📖 [HuggingFace Sentence Transformers](https://www.sbert.net/)
- 📖 [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard) - Rankings de embeddings
- 📖 [Google AI Studio](https://makersuite.google.com/) - Embeddings gratuitos

---

## 📚 Próximo notebook

**Notebook 3: Prompting Avanzado y Técnicas de Prompt Engineering**
- Zero-shot, few-shot, chain-of-thought
- Técnicas para mejorar resultados
- Plantillas y patrones de prompts

---

✨ **¡Felicidades!** Has completado el Notebook 2. Ahora entiendes los fundamentos técnicos que hacen funcionar a los LLMs.

# ðŸ§¬ GuÃ­a de IA Generativa - Embeddings, Tokens y Costes

En este notebook aprenderÃ¡s:
- âœ… QuÃ© son los embeddings y por quÃ© son fundamentales
- âœ… CÃ³mo generar y usar embeddings con APIs gratuitas
- âœ… QuÃ© son los tokens y cÃ³mo se cuentan
- âœ… CÃ³mo calcular costes de APIs y optimizar tu uso
- âœ… Trucos prÃ¡cticos para ahorrar dinero

**ðŸ“š Conocimientos previos necesarios:**
- âœ… Haber completado el Notebook 1 (Conceptos BÃ¡sicos)
- âœ… API key de Gemini configurada
- âœ… Python bÃ¡sico

**âš ï¸ IMPORTANTE - SEGURIDAD:**
- âŒ NUNCA compartas tus API keys en este notebook
- âŒ NO subas este archivo a GitHub si contiene keys
- âœ… Usa variables de entorno o widgets seguros
- âœ… Revoca keys si se exponen accidentalmente

---
**ðŸ“‹ Requisitos previos:**
- Python 3.8+
- Jupyter Notebook o JupyterLab o VS Code
- ConexiÃ³n a internet (para APIs)

# 0. SETUP INICIAL

## ðŸ“š Instalar LibrerÃ­as

Vamos a instalar las librerÃ­as necesarias para trabajar con embeddings, tokens y visualizaciones.

In [1]:
# ðŸ“¦ InstalaciÃ³n de dependencias
# Esto puede tardar 2-3 minutos la primera vez

import sys
import subprocess
print("ðŸ”§ Instalando paquetes necesarios...")
print("=" * 60)

# Lista de paquetes a instalar con sus descripciones
packages = [
    ("google-genai", "ðŸŒŸ Cliente de Google para usar Gemini (GRATIS)"),
    ("sentence-transformers", "ðŸ§¬ Modelos de embeddings locales (sin API)"),
    ("scikit-learn", "ðŸ“Š Para cÃ¡lculos de similitud coseno"),
    ("numpy", "ðŸ”¢ Operaciones numÃ©ricas rÃ¡pidas con arrays"),
    ("pandas", "ðŸ“‹ Manejo de datos en tablas (DataFrames)"),
    ("matplotlib", "ðŸ“ˆ VisualizaciÃ³n de grÃ¡ficos y matrices"),
    ("seaborn", "ðŸŽ¨ Visualizaciones bonitas (mejora matplotlib)"),
    ("plotly", "ðŸŽª GrÃ¡ficos interactivos y 3D"),
    ("ipywidgets", "ðŸŽ¨ Widgets interactivos para inputs y botones"),
    ("tiktoken", "ðŸŽ« Contador oficial de tokens de OpenAI"),
]

for package, descripcion in packages:
    try:
        print(f"ðŸ“¥ Instalando {package}...")
        print(f"   â„¹ï¸  {descripcion}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"âœ… {package} instalado correctamente")
    except Exception as e:
        print(f"âš ï¸ Error instalando {package}: {e}")

print("=" * 60)
print("âœ¨ Â¡InstalaciÃ³n completada! Ejecuta la siguiente celda.")

ðŸ”§ Instalando paquetes necesarios...
ðŸ“¥ Instalando google-genai...
   â„¹ï¸  ðŸŒŸ Cliente de Google para usar Gemini (GRATIS)
âœ… google-genai instalado correctamente
ðŸ“¥ Instalando sentence-transformers...
   â„¹ï¸  ðŸ§¬ Modelos de embeddings locales (sin API)
âœ… sentence-transformers instalado correctamente
ðŸ“¥ Instalando scikit-learn...
   â„¹ï¸  ðŸ“Š Para cÃ¡lculos de similitud coseno
âœ… scikit-learn instalado correctamente
ðŸ“¥ Instalando numpy...
   â„¹ï¸  ðŸ”¢ Operaciones numÃ©ricas rÃ¡pidas con arrays
âœ… numpy instalado correctamente
ðŸ“¥ Instalando pandas...
   â„¹ï¸  ðŸ“‹ Manejo de datos en tablas (DataFrames)
âœ… pandas instalado correctamente
ðŸ“¥ Instalando matplotlib...
   â„¹ï¸  ðŸ“ˆ VisualizaciÃ³n de grÃ¡ficos y matrices
âœ… matplotlib instalado correctamente
ðŸ“¥ Instalando seaborn...
   â„¹ï¸  ðŸŽ¨ Visualizaciones bonitas (mejora matplotlib)
âœ… seaborn instalado correctamente
ðŸ“¥ Instalando plotly...
   â„¹ï¸  ðŸŽª GrÃ¡ficos interactivos y 3D
âœ… pl

## ðŸ“š Importar LibrerÃ­as

In [2]:
# Importaciones estÃ¡ndar
import os
import sys
import json
import time
import warnings
from datetime import datetime
from typing import List, Dict, Optional

# LibrerÃ­as para datos y cÃ¡lculos
import numpy as np
import pandas as pd

# LibrerÃ­as para visualizaciÃ³n
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown, HTML

# LibrerÃ­as para widgets
import ipywidgets as widgets
from ipywidgets import Layout, Button, VBox, HBox, Text, Textarea, Password, Output

# LibrerÃ­as para embeddings y tokens
try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMERS_AVAILABLE = True
except ImportError:
    SENTENCE_TRANSFORMERS_AVAILABLE = False
    print("âš ï¸ sentence-transformers no disponible")

try:
    import tiktoken
    TIKTOKEN_AVAILABLE = True
except ImportError:
    TIKTOKEN_AVAILABLE = False
    print("âš ï¸ tiktoken no disponible")

try:
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("âš ï¸ scikit-learn no disponible")

# Importar Google Gemini
try:
    from google import genai
    from google.genai import types
    GEMINI_AVAILABLE = True
except ImportError:
    GEMINI_AVAILABLE = False
    print("âš ï¸ Google Gemini no disponible (instala con: pip install google-genai)")

# ConfiguraciÃ³n de warnings
warnings.filterwarnings('ignore')

# ConfiguraciÃ³n de visualizaciÃ³n
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# --------------------------------------------------
# âš™ï¸ CONFIGURACIÃ“N SSL PARA PROXIES
# --------------------------------------------------

VERIFICAR_SSL = False  # Cambia a True si no estÃ¡s detrÃ¡s de proxy corporativo

if not VERIFICAR_SSL:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print("âš ï¸ ADVERTENCIA: VerificaciÃ³n SSL desactivada")
    print("   Solo para uso en entornos con proxy")

print("=" * 60)
print("âœ… Todas las librerÃ­as importadas correctamente")
print(f"ðŸ“Š Python version: {sys.version.split()[0]}")
print(f"ðŸ• Fecha actual: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"ðŸ”’ VerificaciÃ³n SSL: {'Activada âœ…' if VERIFICAR_SSL else 'Desactivada âš ï¸'}")
print("=" * 60)

âš ï¸ ADVERTENCIA: VerificaciÃ³n SSL desactivada
   Solo para uso en entornos con proxy
âœ… Todas las librerÃ­as importadas correctamente
ðŸ“Š Python version: 3.12.4
ðŸ• Fecha actual: 2026-03-06 09:55:43
ðŸ”’ VerificaciÃ³n SSL: Desactivada âš ï¸


## ðŸ” ConfiguraciÃ³n de API Keys (Segura)

Usa estos widgets para introducir tu API key de Gemini de forma segura. **Â¡No se mostrarÃ¡ en pantalla!**

### ðŸ†“ Recordatorio: CÃ³mo obtener tu API key de Gemini (GRATIS):
- **Gemini**: https://makersuite.google.com/app/apikey (GRATIS hasta 500 req/dÃ­a)

### ðŸ›¡ï¸ **SEGURIDAD - MUY IMPORTANTE:**

#### Â¿DÃ³nde se guardan tus API keys?
- âœ… Se almacenan en **memoria RAM** usando `os.environ` (variables de entorno)
- âœ… Son **TEMPORALES** - se borran al cerrar el notebook/kernel
- âœ… **NO se guardan** en el archivo `.ipynb`
- âœ… **SEGURO para subir a GitHub**

#### âš ï¸ Riesgos a evitar:
- âŒ NUNCA hagas `print(os.environ['GEMINI_API_KEY'])` y subas el output
- âŒ NO compartas screenshots con keys visibles
- âŒ NUNCA escribas `api_key = "AIza123..."` directamente en cÃ³digo
- âœ… Antes de subir a GitHub, ejecuta "Restart Kernel & Clear All Outputs"
- âœ… Si expones una key por error, **revÃ³cala inmediatamente** desde la plataforma

In [ ]:
# ðŸ” Widget seguro para API Key de Gemini
print("ðŸ”‘ ConfiguraciÃ³n de API Key")
print("=" * 60)
print("")
print("ðŸ›¡ï¸ SEGURIDAD:")
print("  - La key se guarda SOLO EN MEMORIA (os.environ)")
print("  - NO se guarda en el archivo .ipynb")
print("  - Es SEGURO subir este notebook a GitHub despuÃ©s de usarlo")
print("  - Se borrarÃ¡ al cerrar el kernel o notebook")
print("")

# Widget para Gemini
gemini_key_widget = Password(
    placeholder='AIza...',
    description='Gemini:',
    disabled=False,
    layout=Layout(width='500px')
)

# BotÃ³n para guardar configuraciÃ³n
save_button = Button(
    description='ðŸ’¾ Guardar Key',
    button_style='success',
    layout=Layout(width='200px')
)

status_output = Output()

def save_keys(b):
    with status_output:
        status_output.clear_output()
        
        if gemini_key_widget.value and gemini_key_widget.value.startswith('AIza'):
            os.environ['GEMINI_API_KEY'] = gemini_key_widget.value
            print("âœ… Key de Gemini guardada en memoria")
            print("")
            print("ðŸ”’ CONFIRMACIÃ“N DE SEGURIDAD:")
            print("  âœ… Almacenada en: os.environ (memoria RAM)")
            print("  âœ… NO estÃ¡ en el archivo .ipynb")
            print("  âœ… Se borrarÃ¡ al reiniciar kernel")
            print("  âœ… SEGURO para compartir el notebook")
            print("")
            print("âš ï¸ IMPORTANTE: Si cierras la sesiÃ³n, tendrÃ¡s que volver a introducirla.")
        else:
            print("âš ï¸ No se detectÃ³ una key vÃ¡lida de Gemini. Verifica que empiece con 'AIza'.")

save_button.on_click(save_keys)

# Mostrar interfaz
display(VBox([
    widgets.HTML("<h3>ðŸ” Introduce tu API Key de Gemini (GRATIS):</h3>"),
    widgets.HTML("<p style='color: #666;'>ObtÃ©nla en: <a href='https://makersuite.google.com/app/apikey' target='_blank'>makersuite.google.com/app/apikey</a></p>"),
    gemini_key_widget,
    save_button,
    status_output
]))

print("\nðŸ’¡ TIP: TambiÃ©n puedes usar modelos locales sin API key (lo veremos mÃ¡s adelante).")

ðŸ”‘ ConfiguraciÃ³n de API Key

ðŸ›¡ï¸ SEGURIDAD:
  - La key se guarda SOLO EN MEMORIA (os.environ)
  - NO se guarda en el archivo .ipynb
  - Es SEGURO subir este notebook a GitHub despuÃ©s de usarlo
  - Se borrarÃ¡ al cerrar el kernel o notebook




ðŸ’¡ TIP: TambiÃ©n puedes usar modelos locales sin API key (lo veremos mÃ¡s adelante).


## âœ… VerificaciÃ³n del Entorno

Vamos a verificar que todo estÃ© configurado correctamente.

In [4]:
# ðŸ” VerificaciÃ³n del entorno

print("ðŸ” VERIFICACIÃ“N DEL ENTORNO")
print("=" * 60)

# Verificar Python
print(f"âœ… Python: {sys.version.split()[0]}")

# Verificar librerÃ­as principales
print("\nðŸ“š LibrerÃ­as:")
libraries = {
    "google-genai": GEMINI_AVAILABLE,
    "sentence-transformers": SENTENCE_TRANSFORMERS_AVAILABLE,
    "tiktoken": TIKTOKEN_AVAILABLE,
    "scikit-learn": SKLEARN_AVAILABLE,
    "numpy": True,
    "pandas": True,
    "matplotlib": True,
}

for lib_name, lib_status in libraries.items():
    status = "âœ…" if lib_status else "âš ï¸"
    print(f"{status} {lib_name}: {'Disponible' if lib_status else 'No disponible'}")

# Verificar API Key
print("\nðŸ”‘ API Key detectada:")
gemini_key = os.getenv('GEMINI_API_KEY', None)
status = "âœ…" if gemini_key else "âš ï¸"
masked = f"{gemini_key[:8]}...{gemini_key[-4:]}" if gemini_key else "No configurada"
print(f"{status} Gemini: {masked}")

# Verificar conexiÃ³n
print("\nðŸŒ Verificando conexiÃ³n a internet...")
try:
    import requests
    response = requests.get("https://www.google.com", timeout=5, verify=VERIFICAR_SSL)
    print("âœ… ConexiÃ³n a internet: OK")
except:
    print("âš ï¸ Sin conexiÃ³n a internet (necesaria para APIs)")

print("=" * 60)
print("âœ¨ VerificaciÃ³n completada. Â¡EstÃ¡s listo para empezar!")

ðŸ” VERIFICACIÃ“N DEL ENTORNO
âœ… Python: 3.12.4

ðŸ“š LibrerÃ­as:
âœ… google-genai: Disponible
âœ… sentence-transformers: Disponible
âœ… tiktoken: Disponible
âœ… scikit-learn: Disponible
âœ… numpy: Disponible
âœ… pandas: Disponible
âœ… matplotlib: Disponible

ðŸ”‘ API Key detectada:
âœ… Gemini: AIzaSyBh...EBWA

ðŸŒ Verificando conexiÃ³n a internet...
âœ… ConexiÃ³n a internet: OK
âœ¨ VerificaciÃ³n completada. Â¡EstÃ¡s listo para empezar!


## ðŸ“– Recordatorio: Â¿QuÃ© vimos en el Notebook 1?

Antes de continuar, recapitulemos los conceptos clave del Notebook 1:

### âœ… Conceptos que ya conoces:

| Concepto | QuÃ© es | Por quÃ© importa |
|----------|--------|-----------------|
| **IA Generativa** | IA que **crea** contenido nuevo (texto, imÃ¡genes, etc.) | Base de ChatGPT, Gemini, DALL-E |
| **LLM** | **L**arge **L**anguage **M**odel - modelo entrenado con texto masivo | El "cerebro" detrÃ¡s de las IAs conversacionales |
| **API** | Forma de comunicarse con servicios IA remotos | Te permite usar modelos potentes sin hardware caro |
| **Prompt** | InstrucciÃ³n que le das al modelo | La calidad del prompt determina la calidad de la respuesta |
| **Contexto** | Cantidad de informaciÃ³n que el modelo puede "recordar" | LÃ­mite de cuÃ¡nto texto puedes enviar/recibir de una vez |

### ðŸŽ¯ Lo que aprenderÃ¡s HOY:

Ahora vamos **un nivel mÃ¡s profundo**. Hoy aprenderÃ¡s:

1. **ðŸ§¬ Embeddings**: El "idioma secreto" que los LLMs usan internamente para entender el significado
2. **ðŸŽ« Tokens**: La "moneda" con la que se miden y cobran las APIs
3. **ðŸ’° Costes**: CÃ³mo calcular cuÃ¡nto te cuesta usar una API
4. **âš¡ OptimizaciÃ³n**: Trucos para ahorrar dinero

**ðŸ’¡ Piensa en esto como "pasar de usuario a experto".**

En el Notebook 1 aprendiste a **usar** las IAs. Hoy aprenderÃ¡s **cÃ³mo funcionan por dentro** y **cÃ³mo optimizarlas**.

---
# 1. â­ EMBEDDINGS: El idioma secreto de los LLMs

## 1.0 Â¿Por quÃ© los LLMs necesitan embeddings?

### ðŸ¤” El Problema: Los ordenadores no entienden palabras

Piensa en esto:

```
ðŸ‘¨ Humano: "perro" â†’ ðŸ§  Entiendo que es un animal, tiene 4 patas, ladra, es mascota...
ðŸ’» Ordenador: "perro" â†’ â“ Solo ve las letras: p-e-r-r-o
```

**Los ordenadores solo entienden nÃºmeros**, no conceptos.

### âœ¨ La SoluciÃ³n: Embeddings

Un **embedding** es una forma de convertir palabras/frases en **listas de nÃºmeros** (vectores) que capturan su **significado**.

```
"perro" â†’ [0.23, -0.15, 0.89, 0.12, ..., -0.34]  (768 nÃºmeros)
"gato"  â†’ [0.25, -0.18, 0.87, 0.09, ..., -0.31]  (768 nÃºmeros)
"coche" â†’ [-0.67, 0.92, -0.12, 0.45, ..., 0.78]  (768 nÃºmeros)
```

### ðŸ§­ La Magia: Palabras similares tienen nÃºmeros similares

```
Distancia entre embeddings:
"perro" â†” "gato"  = 0.15  (MUY CERCANOS - ambos son animales)
"perro" â†” "coche" = 0.92  (MUY LEJANOS - no tienen relaciÃ³n)
```

### ðŸ“Š VisualizaciÃ³n Conceptual

Imagina que cada palabra vive en un **espacio 3D**:

```
        ðŸ• perro
         â”‚
         â”‚ (cercanos)
         â”‚
        ðŸˆ gato
        
        
        
        
        ðŸš— coche (lejos)
```

**En realidad**, los embeddings tienen **cientos o miles de dimensiones**, no solo 3, pero la idea es la misma: **palabras con significados similares estÃ¡n cerca en el "espacio de embeddings"**.

---

### ðŸŽ¯ Â¿Para quÃ© se usan los embeddings?

| AplicaciÃ³n | CÃ³mo funciona | Ejemplo real |
|------------|---------------|--------------|
| **BÃºsqueda semÃ¡ntica** | Encuentra documentos por significado, no por palabra exacta | Google, ChatGPT con documentos |
| **Recomendaciones** | "Si te gustÃ³ X, te gustarÃ¡ Y" | Netflix, Spotify, Amazon |
| **ClasificaciÃ³n** | Agrupar textos similares | Filtro de spam, anÃ¡lisis de sentimiento |
| **Chatbots** | Entender intenciÃ³n del usuario | Asistentes virtuales, soporte tÃ©cnico |
| **TraducciÃ³n** | Capturar significado entre idiomas | Google Translate, DeepL |

---

### ðŸ’¡ Conceptos Clave para Recordar:

1. **Embedding = Vector numÃ©rico que representa el significado**
2. **DimensiÃ³n = Cantidad de nÃºmeros en el vector** (tÃ­picamente 128, 384, 768, 1536...)
3. **Similitud = Palabras/frases similares â†’ vectores cercanos**
4. **Los embeddings se crean con modelos especializados** (no son aleatorios)

---

### ðŸŽ“ Ejemplo Real vs Tradicional

#### âŒ BÃºsqueda tradicional (por palabra exacta):
```
Usuario busca: "vehÃ­culo rÃ¡pido"
Base de datos: "coche deportivo"
Resultado: âŒ NO ENCUENTRA (palabras diferentes)
```

#### âœ… BÃºsqueda con embeddings (por significado):
```
Usuario busca: "vehÃ­culo rÃ¡pido" â†’ embedding [0.45, -0.23, ...]
Base de datos: "coche deportivo" â†’ embedding [0.47, -0.25, ...]
Similitud: 0.95 (muy similar)
Resultado: âœ… ENCUENTRA (significado igual)
```

---

**ðŸŽ¯ Resumen en una frase:**

*Los embeddings son como un "traductor universal" que convierte cualquier texto a nÃºmeros, permitiendo a los ordenadores "entender" el significado y comparar textos de forma inteligente.*

### ðŸ§¬ Cargar Modelo Local (Opcional)

**Nota importante:** El modelo local es **OPCIONAL**. Si falla la descarga:
- âœ… El notebook **seguirÃ¡ funcionando** con la API de Gemini
- âœ… AprenderÃ¡s los mismos conceptos
- âœ… PodrÃ¡s hacer todos los ejercicios

**Ventajas del modelo local:**
- ðŸ†“ Gratis e ilimitado
- ðŸ”’ 100% privado (offline)
- âš¡ InstantÃ¡neo (no esperas respuesta API)

In [13]:
# ðŸ§¬ Generador de Embeddings Local (OPCIONAL)
print("ðŸ”„ Intentando cargar modelo de embeddings local...")
print("   (Primera vez puede tardar 1-2 minutos descargando el modelo)")
print("   âš ï¸ Si falla, no te preocupes - usaremos API de Gemini")
print("=" * 60)

model = None  # Inicializar como None

if SENTENCE_TRANSFORMERS_AVAILABLE:
    # Configurar SSL para HuggingFace si es necesario
    if not VERIFICAR_SSL:
        print("âš ï¸ Configurando descarga sin verificaciÃ³n SSL...")
        import ssl
        
        # Configurar a nivel de SSL
        try:
            ssl._create_default_https_context = ssl._create_unverified_context
        except:
            pass
    
    # Cargar modelo pequeÃ±o y rÃ¡pido (33MB)
    # 'paraphrase-multilingual-MiniLM-L12-v2' - Soporta 50+ idiomas incluyendo espaÃ±ol
    model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
    
    try:
        model = SentenceTransformer(
            model_name,
            trust_remote_code=True
        )
        print(f"âœ… Modelo local cargado exitosamente: {model_name}")
        print(f"ðŸ“ DimensiÃ³n de embeddings: {model.get_sentence_embedding_dimension()}")
        print(f"ðŸŒ Soporta: EspaÃ±ol, InglÃ©s, FrancÃ©s, AlemÃ¡n, y 46+ idiomas")
        print("=" * 60)
        
        # Generar embedding de una frase de ejemplo
        frase_ejemplo = "Los perros son animales leales"
        embedding = model.encode(frase_ejemplo)
        
        print(f"\nðŸ“ Frase: '{frase_ejemplo}'")
        print(f"ðŸ§¬ Embedding generado:")
        print(f"   - Tipo: {type(embedding)}")
        print(f"   - Forma: {embedding.shape}")
        print(f"   - Dimensiones: {len(embedding)} nÃºmeros")
        print(f"\n   Primeros 10 nÃºmeros del embedding:")
        print(f"   {embedding[:10]}")
        print(f"\n   ðŸ’¡ En total son {len(embedding)} nÃºmeros que representan el significado de la frase")
        
    except Exception as e:
        print(f"âš ï¸ No se pudo cargar el modelo local (esperado en entornos con proxy)")
        print(f"   Error tÃ©cnico: {str(e)[:100]}...")
        print("\n   ðŸŽ¯ NO TE PREOCUPES - Esto es NORMAL en entornos corporativos")
        print("\n   âœ… SOLUCIÃ“N: Continuaremos usando la API de Gemini")
        print("      - FuncionarÃ¡ igual de bien")
        print("      - AprenderÃ¡s los mismos conceptos")
        print("      - Los ejercicios funcionarÃ¡n perfectamente")
        print("\n   ðŸ’¡ OPCIONAL: Si quieres el modelo local mÃ¡s adelante:")
        print("      1. DescÃ¡rgalo fuera del proxy corporativo")
        print("      2. O configura HTTP_PROXY/HTTPS_PROXY en la celda anterior")
        model = None
else:
    print("âš ï¸ sentence-transformers no estÃ¡ instalado")
    print("   ðŸ’¡ No es problema - usaremos API de Gemini")
    model = None

if model is None:
    print("\n" + "=" * 60)
    print("ðŸŒŸ MODO: API de Gemini (sin modelo local)")
    print("âœ… El notebook funcionarÃ¡ perfectamente")
    print("=" * 60)
else:
    print("\n" + "=" * 60)
    print("ðŸŒŸ MODO: Modelo Local + API de Gemini")
    print("âœ… Tienes lo mejor de ambos mundos")
    print("=" * 60)

ðŸ”„ Intentando cargar modelo de embeddings local...
   (Primera vez puede tardar 1-2 minutos descargando el modelo)
   âš ï¸ Si falla, no te preocupes - usaremos API de Gemini
âš ï¸ Configurando descarga sin verificaciÃ³n SSL...


'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
No sentence-transformers model found with name sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2. Creating a new one with mean pooling.
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


âš ï¸ No se pudo cargar el modelo local (esperado en entornos con proxy)
   Error tÃ©cnico: Cannot send a request, as the client has been closed....

   ðŸŽ¯ NO TE PREOCUPES - Esto es NORMAL en entornos corporativos

   âœ… SOLUCIÃ“N: Continuaremos usando la API de Gemini
      - FuncionarÃ¡ igual de bien
      - AprenderÃ¡s los mismos conceptos
      - Los ejercicios funcionarÃ¡n perfectamente

   ðŸ’¡ OPCIONAL: Si quieres el modelo local mÃ¡s adelante:
      1. DescÃ¡rgalo fuera del proxy corporativo
      2. O configura HTTP_PROXY/HTTPS_PROXY en la celda anterior

ðŸŒŸ MODO: API de Gemini (sin modelo local)
âœ… El notebook funcionarÃ¡ perfectamente


### ðŸŽ® Widget Interactivo: Genera tus propios embeddings

Escribe cualquier frase y ve su embedding en tiempo real:

In [14]:
# ðŸŽ® Widget interactivo para generar embeddings

if model is not None:
    # Widgets
    input_text = Textarea(
        placeholder='Escribe cualquier frase aquÃ­... (espaÃ±ol, inglÃ©s, etc.)',
        description='Tu frase:',
        disabled=False,
        layout=Layout(width='600px', height='80px')
    )
    
    generate_button = Button(
        description='ðŸ§¬ Generar Embedding',
        button_style='info',
        layout=Layout(width='200px')
    )
    
    output_area = Output()
    
    def generate_embedding(b):
        with output_area:
            output_area.clear_output()
            
            if not input_text.value.strip():
                print("âš ï¸ Por favor, escribe una frase primero")
                return
            
            print("ðŸ”„ Generando embedding...")
            frase = input_text.value.strip()
            emb = model.encode(frase)
            
            print("=" * 60)
            print(f"ðŸ“ Frase: '{frase}'")
            print(f"ðŸ“ Longitud de la frase: {len(frase)} caracteres, {len(frase.split())} palabras")
            print(f"\nðŸ§¬ Embedding generado:")
            print(f"   - Dimensiones: {len(emb)} nÃºmeros")
            print(f"   - Rango de valores: [{emb.min():.3f}, {emb.max():.3f}]")
            print(f"   - Media: {emb.mean():.3f}")
            print(f"\n   Primeros 20 nÃºmeros:")
            print(f"   {emb[:20]}")
            print(f"\n   ðŸ’¡ Nota: Estos {len(emb)} nÃºmeros capturan todo el SIGNIFICADO de tu frase")
            print("=" * 60)
    
    generate_button.on_click(generate_embedding)
    
    # Mostrar interfaz
    display(VBox([
        widgets.HTML("<h3>ðŸŽ® Generador de Embeddings Interactivo</h3>"),
        widgets.HTML("<p style='color: #666;'>Escribe cualquier frase y presiona el botÃ³n para ver su embedding:</p>"),
        input_text,
        generate_button,
        output_area
    ]))
    
else:
    print("â„¹ï¸ Generador con modelo local no disponible")
    print("   ðŸ’¡ No te preocupes - en la siguiente secciÃ³n usaremos API de Gemini")
    print("   âœ… AprenderÃ¡s exactamente lo mismo")

â„¹ï¸ Generador con modelo local no disponible
   ðŸ’¡ No te preocupes - en la siguiente secciÃ³n usaremos API de Gemini
   âœ… AprenderÃ¡s exactamente lo mismo


## 1.2 AnalogÃ­a Visual: Mapa SemÃ¡ntico 2D

Los embeddings reales tienen cientos de dimensiones, pero podemos usar tÃ©cnicas de reducciÃ³n dimensional para **visualizarlos en 2D** y ver cÃ³mo se agrupan las palabras similares.

### ðŸ—ºï¸ Vamos a crear un "mapa de palabras"

In [15]:
# ðŸ—ºï¸ VisualizaciÃ³n de embeddings en 2D

if model is not None and SKLEARN_AVAILABLE:
    from sklearn.decomposition import PCA
    
    # Lista de palabras de diferentes categorÃ­as
    palabras = [
        # Animales domÃ©sticos
        "perro", "gato", "canario", "hÃ¡mster",
        # VehÃ­culos
        "coche", "moto", "bicicleta", "camiÃ³n",
        # Frutas
        "manzana", "naranja", "plÃ¡tano", "fresa",
        # Profesiones
        "mÃ©dico", "profesor", "ingeniero", "abogado",
        # Deportes
        "fÃºtbol", "tenis", "nataciÃ³n", "baloncesto"
    ]
    
    print("ðŸ§¬ Generando embeddings de 20 palabras...")
    embeddings = model.encode(palabras)
    print(f"âœ… {len(embeddings)} embeddings generados ({embeddings.shape[1]} dimensiones cada uno)")
    
    print("\nðŸ”„ Reduciendo de {embeddings.shape[1]} dimensiones a 2D para visualizaciÃ³n...")
    # PCA: ReducciÃ³n dimensional de 384D â†’ 2D (preservando mÃ¡xima informaciÃ³n)
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(embeddings)
    print(f"âœ… ReducciÃ³n completada (conserva {pca.explained_variance_ratio_.sum()*100:.1f}% de la informaciÃ³n)")
    
    # Crear DataFrame para Plotly
    df_plot = pd.DataFrame({
        'x': embeddings_2d[:, 0],
        'y': embeddings_2d[:, 1],
        'palabra': palabras,
        'categoria': ['Animales']*4 + ['VehÃ­culos']*4 + ['Frutas']*4 + ['Profesiones']*4 + ['Deportes']*4
    })
    
    # Crear grÃ¡fico interactivo
    fig = px.scatter(
        df_plot, 
        x='x', 
        y='y', 
        text='palabra',
        color='categoria',
        title='ðŸ—ºï¸ Mapa SemÃ¡ntico: Palabras en el Espacio de Embeddings',
        labels={'x': 'DimensiÃ³n 1', 'y': 'DimensiÃ³n 2'},
        width=900,
        height=600
    )
    
    # Ajustar diseÃ±o
    fig.update_traces(
        textposition='top center',
        marker=dict(size=12, line=dict(width=2, color='white'))
    )
    
    fig.update_layout(
        font=dict(size=12),
        showlegend=True,
        hovermode='closest'
    )
    
    fig.show()
    
    print("\nðŸ’¡ OBSERVA:")
    print("   âœ… Palabras de la misma categorÃ­a estÃ¡n CERCA (mismo color)")
    print("   âœ… Palabras de categorÃ­as diferentes estÃ¡n LEJOS")
    print("   âœ… Esto demuestra que los embeddings capturan el SIGNIFICADO")
    print("\n   ðŸŽ¯ El modelo NUNCA vio que 'perro' y 'gato' son animales")
    print("      Lo aprendiÃ³ leyendo millones de textos donde aparecen en contextos similares")
    
else:
    print("âš ï¸ VisualizaciÃ³n no disponible (requiere modelo y scikit-learn)")

âš ï¸ VisualizaciÃ³n no disponible (requiere modelo y scikit-learn)


## 1.3 Ejercicio: Predice Similitudes

Antes de calcular computacionalmente, **intenta adivinar** quÃ© pares de frases serÃ¡n mÃ¡s similares.

### ðŸŽ¯ Instrucciones:
1. Lee las 4 frases a continuaciÃ³n
2. Piensa: Â¿cuÃ¡les tienen significados mÃ¡s cercanos?
3. OrdÃ©nalas mentalmente de **mÃ¡s similar a menos similar**
4. Luego ejecuta la siguiente celda para ver si acertaste

### ðŸ“ Las 4 frases:

| Frase | Texto |
|-------|-------|
| **A** | "El perro juega en el parque" |
| **B** | "Un cachorro se divierte al aire libre" |
| **C** | "Me gusta comer pizza los viernes" |
| **D** | "Python es un lenguaje de programaciÃ³n" |

### ðŸ¤” Pregunta:
- Â¿QuÃ© par de frases crees que tendrÃ¡ la similitud MÃS ALTA?
- Â¿QuÃ© par tendrÃ¡ la similitud MÃS BAJA?

**ðŸ’­ PiÃ©nsalo antes de continuar...**

In [ ]:
# ðŸŽ¯ CÃ¡lculo de Similitudes entre Frases

if model is not None and SKLEARN_AVAILABLE:
    # Las 4 frases del ejercicio
    frases = [
        "El perro juega en el parque",           # A
        "Un cachorro se divierte al aire libre",  # B
        "Me gusta comer pizza los viernes",      # C
        "Python es un lenguaje de programaciÃ³n"  # D
    ]
    
    labels = ["A", "B", "C", "D"]
    
    print("ðŸ§¬ Generando embeddings de las 4 frases...")
    frases_embeddings = model.encode(frases)
    print("âœ… Embeddings generados\n")
    
    print("ðŸ“Š Calculando similitudes (similitud coseno)...")
    # Similitud coseno: valor entre -1 y 1
    # 1.0 = idÃ©nticos, 0.0 = no relacionados, -1.0 = opuestos
    similitudes = cosine_similarity(frases_embeddings)
    
    # Crear DataFrame para mejor visualizaciÃ³n
    df_sim = pd.DataFrame(similitudes, index=labels, columns=labels)
    
    print("\nðŸ”¢ MATRIZ DE SIMILITUDES:")
    print("=" * 60)
    print(df_sim.round(3))
    print("=" * 60)
    
    # Encontrar pares mÃ¡s y menos similares (excluyendo diagonal)
    max_sim = 0
    min_sim = 1
    max_pair = ("", "")
    min_pair = ("", "")
    
    for i in range(len(frases)):
        for j in range(i+1, len(frases)):
            sim = similitudes[i][j]
            if sim > max_sim:
                max_sim = sim
                max_pair = (labels[i], labels[j])
            if sim < min_sim:
                min_sim = sim
                min_pair = (labels[i], labels[j])
    
    print(f"\nðŸ† PAR MÃS SIMILAR:")
    print(f"   {max_pair[0]} â†” {max_pair[1]}: {max_sim:.3f}")
    print(f"   '{frases[labels.index(max_pair[0])]}'")
    print(f"   '{frases[labels.index(max_pair[1])]}'")
    
    print(f"\nâŒ PAR MENOS SIMILAR:")
    print(f"   {min_pair[0]} â†” {min_pair[1]}: {min_sim:.3f}")
    print(f"   '{frases[labels.index(min_pair[0])]}'")
    print(f"   '{frases[labels.index(min_pair[1])]}'")
    
    print("\nðŸ’¡ INTERPRETACIÃ“N:")
    print("   - Similitud > 0.7: MUY similares (mismo tema/significado)")
    print("   - Similitud 0.3-0.7: Algo relacionadas")
    print("   - Similitud < 0.3: Poco o nada relacionadas")
    
    print("\nðŸŽ¯ Â¿ACERTASTE?")
    print(f"   - Las frases A y B (perro/cachorro) deberÃ­an ser las MÃS similares")
    print(f"   - Las frases B y D (cachorro/Python) deberÃ­an ser las MENOS similares")
    
    # VisualizaciÃ³n con mapa de calor
    print("\nðŸ“ˆ Generando mapa de calor...")
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_sim, annot=True, fmt='.3f', cmap='coolwarm', 
                vmin=0, vmax=1, square=True, cbar_kws={'label': 'Similitud'})
    plt.title('ðŸ”¥ Mapa de Calor: Similitud entre Frases', fontsize=14, fontweight='bold')
    plt.xlabel('Frase', fontsize=12)
    plt.ylabel('Frase', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    print("âœ… Mapa de calor generado")
    print("   ðŸ’¡ Colores cÃ¡lidos (rojo) = MÃS similar")
    print("   ðŸ’¡ Colores frÃ­os (azul) = MENOS similar")
    
else:
    print("â„¹ï¸ Ejercicio con modelo local no disponible")
    print("   ðŸ’¡ Continuaremos con API de Gemini en la siguiente secciÃ³n")
    print("   âœ… AprenderÃ¡s el mismo concepto de similitud mÃ¡s adelante")

---
# 2. EMBEDDINGS EN ACCIÃ“N

## 2.1 Generador de Embeddings con API de Gemini (GRATIS)

Ahora vamos a usar la API de **Google Gemini** para generar embeddings. Gemini ofrece embeddings GRATIS con su modelo **text-embedding-004**.

### ðŸ†š Modelo Local vs API de Gemini

| CaracterÃ­stica | Modelo Local (Sentence-Transformers) | API Gemini |
|----------------|--------------------------------------|------------|
| **Dimensiones** | 384 | 768 |
| **Calidad** | Buena | Excelente (state-of-the-art) |
| **Velocidad** | RÃ¡pida | Muy rÃ¡pida |
| **Coste** | ðŸ†“ Gratis (corre en tu PC) | ðŸ†“ Gratis (hasta 500 req/dÃ­a) |
| **Idiomas** | 50+ idiomas | 100+ idiomas |
| **Internet** | âŒ No necesita | âœ… Necesita |
| **Privacidad** | âœ… 100% local | âš ï¸ Datos van a Google |

Vamos a probarlo:

In [ ]:
# ðŸŒŸ Generar Embeddings con API de Gemini

gemini_key = os.getenv('GEMINI_API_KEY', None)

if gemini_key and GEMINI_AVAILABLE:
    print("ðŸ”„ Configurando cliente de Gemini...")
    
    try:
        # Configurar cliente
        client = genai.Client(api_key=gemini_key)
        
        # Texto de prueba
        texto_prueba = "La inteligencia artificial estÃ¡ transformando el mundo"
        
        print(f"ðŸ“ Texto: '{texto_prueba}'")
        print("\nðŸŒ Enviando peticiÃ³n a API de Gemini...")
        
        # Generar embedding
        response = client.models.embed_content(
            model='models/text-embedding-004',
            contents=[texto_prueba]
        )
        
        # Extraer embedding
        embedding_gemini = np.array(response['embedding'])
        
        print("âœ… Embedding recibido desde Gemini")
        print("=" * 60)
        print(f"ðŸ§¬ InformaciÃ³n del embedding:")
        print(f"   - Modelo: text-embedding-004")
        print(f"   - Dimensiones: {len(embedding_gemini)} nÃºmeros")
        print(f"   - Rango: [{embedding_gemini.min():.4f}, {embedding_gemini.max():.4f}]")
        print(f"   - Media: {embedding_gemini.mean():.4f}")
        print(f"\n   Primeros 20 valores:")
        print(f"   {embedding_gemini[:20]}")
        print("=" * 60)
        
        # Comparar con modelo local (si estÃ¡ disponible)
        if model is not None:
            embedding_local = model.encode(texto_prueba)
            
            print("\nðŸ“Š COMPARACIÃ“N: Gemini vs Local")
            print("=" * 60)
            print(f"{'MÃ©trica':<25} {'Gemini':>15} {'Local':>15}")
            print("-" * 60)
            print(f"{'Dimensiones':<25} {len(embedding_gemini):>15} {len(embedding_local):>15}")
            print(f"{'Rango mÃ­nimo':<25} {embedding_gemini.min():>15.4f} {embedding_local.min():>15.4f}")
            print(f"{'Rango mÃ¡ximo':<25} {embedding_gemini.max():>15.4f} {embedding_local.max():>15.4f}")
            print(f"{'Media':<25} {embedding_gemini.mean():>15.4f} {embedding_local.mean():>15.4f}")
            print("=" * 60)
            
            print("\nðŸ’¡ NOTA:")
            print("   - Gemini tiene MÃS dimensiones (768 vs 384) = mÃ¡s informaciÃ³n")
            print("   - Ambos capturan el significado, pero Gemini suele ser mÃ¡s preciso")
            print("   - Para producciÃ³n, Gemini es recomendado (gratis + alta calidad)")
        
        print("\nâœ… Â¡Gemini API funcionando correctamente!")
        
    except Exception as e:
        print(f"âŒ Error al usar Gemini API: {e}")
        print("\nðŸ” Posibles causas:")
        print("   1. API Key incorrecta")
        print("   2. Sin conexiÃ³n a internet")
        print("   3. LÃ­mite de cuota excedido (500 req/dÃ­a)")
        print("   4. Verificar SSL desactivado (si estÃ¡s en proxy)")
        
else:
    if not gemini_key:
        print("âš ï¸ API Key de Gemini no configurada")
        print("   SoluciÃ³n: Ejecuta la celda de configuraciÃ³n de API Keys")
    if not GEMINI_AVAILABLE:
        print("âš ï¸ LibrerÃ­a google-genai no disponible")
        print("   SoluciÃ³n: pip install google-genai")

## 2.2 Calculadora de Similitudes Coseno

Ahora que sabemos generar embeddings, vamos a crear una **calculadora de similitudes** interactiva.

### ðŸ“ Â¿QuÃ© es la Similitud Coseno?

La **similitud coseno** mide el Ã¡ngulo entre dos vectores de embeddings:

```
Similitud Coseno = cos(Î¸) donde Î¸ es el Ã¡ngulo entre vectores

Valores:
  1.0  = Vectores idÃ©nticos (Î¸ = 0Â°)
  0.5  = Vectores con Ã¡ngulo de 60Â°
  0.0  = Vectores perpendiculares (Î¸ = 90Â°, sin relaciÃ³n)
 -1.0  = Vectores opuestos (Î¸ = 180Â°)
```

**FÃ³rmula matemÃ¡tica:**

$$
\text{similitud} = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{||\mathbf{A}|| \times ||\mathbf{B}||}
$$

Donde:
- $\mathbf{A}$ y $\mathbf{B}$ son los vectores de embeddings
- $\mathbf{A} \cdot \mathbf{B}$ es el producto escalar
- $||\mathbf{A}||$ es la norma (longitud) del vector A

**ðŸ’¡ En tÃ©rminos simples:** compara la "direcciÃ³n" de dos vectores en el espacio.

In [ ]:
# ðŸ§® Calculadora Interactiva de Similitud

if model is not None and SKLEARN_AVAILABLE:
    
    # Widgets
    texto1_widget = Textarea(
        placeholder='Escribe la primera frase...',
        description='Frase 1:',
        layout=Layout(width='600px', height='60px')
    )
    
    texto2_widget = Textarea(
        placeholder='Escribe la segunda frase...',
        description='Frase 2:',
        layout=Layout(width='600px', height='60px')
    )
    
    calcular_button = Button(
        description='ðŸ§® Calcular Similitud',
        button_style='success',
        layout=Layout(width='200px')
    )
    
    resultado_output = Output()
    
    def calcular_similitud(b):
        with resultado_output:
            resultado_output.clear_output()
            
            if not texto1_widget.value.strip() or not texto2_widget.value.strip():
                print("âš ï¸ Por favor, escribe ambas frases primero")
                return
            
            frase1 = texto1_widget.value.strip()
            frase2 = texto2_widget.value.strip()
            
            print("ðŸ§¬ Generando embeddings...")
            emb1 = model.encode([frase1])[0]
            emb2 = model.encode([frase2])[0]
            
            print("ðŸ§® Calculando similitud coseno...")
            similitud = cosine_similarity([emb1], [emb2])[0][0]
            
            print("=" * 60)
            print(f"ðŸ“ Frase 1: '{frase1}'")
            print(f"ðŸ“ Frase 2: '{frase2}'")
            print(f"\nðŸŽ¯ SIMILITUD COSENO: {similitud:.4f}")
            print("=" * 60)
            
            # InterpretaciÃ³n
            if similitud >= 0.8:
                interpretacion = "ðŸŸ¢ MUY SIMILARES - Significado casi idÃ©ntico"
                emoji = "ðŸŽ‰"
            elif similitud >= 0.6:
                interpretacion = "ðŸŸ¡ BASTANTE SIMILARES - Hablan de temas relacionados"
                emoji = "ðŸ‘"
            elif similitud >= 0.4:
                interpretacion = "ðŸŸ  ALGO SIMILARES - Tienen alguna conexiÃ³n"
                emoji = "ðŸ¤”"
            elif similitud >= 0.2:
                interpretacion = "ðŸ”´ POCO SIMILARES - Temas diferentes"
                emoji = "ðŸ¤·"
            else:
                interpretacion = "âš« NADA SIMILARES - No tienen relaciÃ³n"
                emoji = "âŒ"
            
            print(f"\n{emoji} INTERPRETACIÃ“N:")
            print(f"   {interpretacion}")
            
            # Barra visual
            print(f"\nðŸ“Š VISUALIZACIÃ“N:")
            barra_longitud = 50
            lleno = int(similitud * barra_longitud)
            vacio = barra_longitud - lleno
            barra = "â–ˆ" * lleno + "â–‘" * vacio
            print(f"   0.0 [{barra}] 1.0")
            print(f"        {similitud:.4f}")
            
            print("\nðŸ’¡ ESCALA DE REFERENCIA:")
            print("   1.00 - 0.80: Muy similar")
            print("   0.80 - 0.60: Bastante similar")
            print("   0.60 - 0.40: Algo similar")
            print("   0.40 - 0.20: Poco similar")
            print("   0.20 - 0.00: Nada similar")
    
    calcular_button.on_click(calcular_similitud)
    
    # Interfaz
    display(VBox([
        widgets.HTML("<h3>ðŸ§® Calculadora de Similitud SemÃ¡ntica</h3>"),
        widgets.HTML("<p style='color: #666;'>Introduce dos frases y calcula quÃ© tan similares son sus significados:</p>"),
        texto1_widget,
        texto2_widget,
        calcular_button,
        resultado_output
    ]))
    
    # Ejemplos sugeridos
    display(widgets.HTML("""
        <div style='margin-top: 20px; padding: 15px; background-color: #f0f0f0; border-radius: 5px;'>
            <b>ðŸ’¡ Prueba estos ejemplos:</b><br>
            <b>Par 1 (muy similar):</b><br>
            â€¢ "El gato duerme en el sofÃ¡"<br>
            â€¢ "Un felino descansa en el sillÃ³n"<br><br>
            <b>Par 2 (poco similar):</b><br>
            â€¢ "Me encanta la programaciÃ³n"<br>
            â€¢ "El sol brilla en verano"
        </div>
    """))
    
else:
    print("â„¹ï¸ Calculadora con modelo local no disponible")
    print("   ðŸ’¡ ContinuarÃ¡s aprendiendo sobre similitud con API de Gemini")
    print("   âœ… El concepto es el mismo, solo cambia la herramienta")

## 2.3 VisualizaciÃ³n: Matriz de Parecidos

Vamos a crear una **matriz de similitudes** para comparar mÃºltiples frases a la vez.

In [ ]:
# ðŸ“Š Matriz de Similitudes MÃºltiples

if model is not None and SKLEARN_AVAILABLE:
    
    # Lista de frases variadas
    frases_ejemplo = [
        "Me gusta programar en Python",
        "Disfruto escribir cÃ³digo en JavaScript",
        "El sol brilla en el cielo azul",
        "Hace un dÃ­a soleado y despejado",
        "Los perros son animales leales",
        "Los gatos son mascotas independientes",
        "La inteligencia artificial avanza rÃ¡pido",
        "El machine learning es fascinante"
    ]
    
    print("ðŸ§¬ Generando embeddings de 8 frases...")
    frases_embs = model.encode(frases_ejemplo)
    print("âœ… Embeddings generados\n")
    
    print("ðŸ§® Calculando matriz de similitudes...")
    matriz_sim = cosine_similarity(frases_embs)
    print("âœ… Matriz calculada\n")
    
    # Crear labels cortos para visualizaciÃ³n
    labels_cortos = [
        "Python",
        "JavaScript",
        "Sol brillante",
        "DÃ­a soleado",
        "Perros leales",
        "Gatos independientes",
        "IA avanza",
        "ML fascinante"
    ]
    
    # Crear DataFrame
    df_matriz = pd.DataFrame(matriz_sim, index=labels_cortos, columns=labels_cortos)
    
    print("ðŸ“Š MATRIZ DE SIMILITUDES:")
    print("=" * 60)
    print(df_matriz.round(3))
    print("=" * 60)
    
    # VisualizaciÃ³n con mapa de calor mejorado
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        df_matriz, 
        annot=True, 
        fmt='.2f', 
        cmap='RdYlGn',  # Rojo-Amarillo-Verde
        vmin=0, 
        vmax=1, 
        square=True,
        cbar_kws={'label': 'Similitud Coseno'},
        linewidths=0.5,
        linecolor='gray'
    )
    plt.title('ðŸ”¥ Matriz de Similitudes: 8 Frases Diferentes', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Frase', fontsize=12, fontweight='bold')
    plt.ylabel('Frase', fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    print("\nðŸ’¡ OBSERVACIONES:")
    print("   ðŸŸ¢ Verde oscuro (0.8-1.0): Frases muy similares")
    print("   ðŸŸ¡ Amarillo (0.4-0.7): Frases relacionadas")
    print("   ðŸ”´ Rojo (0.0-0.3): Frases sin relaciÃ³n")
    print("\nðŸŽ¯ PATRONES DETECTADOS:")
    print("   - 'Python' y 'JavaScript' son muy similares (ambos sobre programaciÃ³n)")
    print("   - 'Sol brillante' y 'DÃ­a soleado' son muy similares (mismo tema)")
    print("   - 'IA avanza' y 'ML fascinante' son muy similares (mismo campo)")
    print("   - 'Perros leales' y 'Gatos independientes' son similares (ambos mascotas)")
    
else:
    print("â„¹ï¸ VisualizaciÃ³n con modelo local no disponible")
    print("   ðŸ’¡ MÃ¡s adelante verÃ¡s similitudes con API de Gemini")
    print("   âœ… El concepto de matriz de similitud se aplica igual")

## 2.4 Mini Buscador SemÃ¡ntico Simple

Ahora vamos a construir un **buscador semÃ¡ntico** bÃ¡sico. 

### ðŸ” Â¿QuÃ© es un Buscador SemÃ¡ntico?

A diferencia de Google tradicional que busca **palabras exactas**, un buscador semÃ¡ntico busca por **significado**.

**Ejemplo:**

| BÃºsqueda | Buscador Tradicional | Buscador SemÃ¡ntico |
|----------|----------------------|--------------------|
| "vehÃ­culo rÃ¡pido" | âŒ Solo encuentra documentos con "vehÃ­culo" y "rÃ¡pido" | âœ… Encuentra "coche deportivo", "auto veloz", "Ferrari", etc. |

Vamos a crear uno bÃ¡sico donde tÃº aportas los documentos:

In [ ]:
# ðŸ” Mini Buscador SemÃ¡ntico

if model is not None and SKLEARN_AVAILABLE:
    
    # Base de datos de documentos de ejemplo
    documentos = [
        "Python es un lenguaje de programaciÃ³n muy popular para ciencia de datos",
        "El cafÃ© tiene cafeÃ­na que te mantiene despierto",
        "Los perros son animales leales y excelentes compaÃ±eros",
        "Machine learning es una rama de la inteligencia artificial",
        "La Torre Eiffel estÃ¡ en ParÃ­s, Francia",
        "JavaScript se usa principalmente para desarrollo web",
        "El chocolate tiene propiedades antioxidantes beneficiosas",
        "Los gatos son mascotas independientes que cazan ratones",
        "Deep learning usa redes neuronales profundas",
        "La pasta italiana es muy popular en todo el mundo",
        "React es una biblioteca de JavaScript para interfaces",
        "El tÃ© verde es bueno para la salud cardiovascular",
        "Los leones son los reyes de la selva africana",
        "TensorFlow es un framework para machine learning",
        "El Coliseo Romano es un antiguo anfiteatro"
    ]
    
    print("ðŸ”„ Preparando buscador semÃ¡ntico...")
    print(f"   ðŸ“š Base de datos: {len(documentos)} documentos")
    print("\nðŸ§¬ Generando embeddings de todos los documentos...")
    documentos_embs = model.encode(documentos)
    print("âœ… Base de datos indexada\n")
    
    # Widget de bÃºsqueda
    busqueda_widget = Text(
        placeholder='Escribe tu bÃºsqueda aquÃ­... (ej: "programaciÃ³n", "animales", "comida")',
        description='Buscar:',
        layout=Layout(width='600px')
    )
    
    top_k_widget = widgets.IntSlider(
        value=3,
        min=1,
        max=10,
        step=1,
        description='Resultados:',
        layout=Layout(width='400px')
    )
    
    buscar_button = Button(
        description='ðŸ” Buscar',
        button_style='primary',
        layout=Layout(width='150px')
    )
    
    resultados_output = Output()
    
    def realizar_busqueda(b):
        with resultados_output:
            resultados_output.clear_output()
            
            if not busqueda_widget.value.strip():
                print("âš ï¸ Por favor, escribe algo para buscar")
                return
            
            query = busqueda_widget.value.strip()
            top_k = top_k_widget.value
            
            print(f"ðŸ” Buscando: '{query}'")
            print(f"   ðŸ“Š Mostrando top {top_k} resultados\n")
            
            # Generar embedding de la consulta
            query_emb = model.encode([query])[0]
            
            # Calcular similitudes con todos los documentos
            similitudes = cosine_similarity([query_emb], documentos_embs)[0]
            
            # Ordenar por similitud (mayor a menor)
            indices_ordenados = np.argsort(similitudes)[::-1][:top_k]
            
            print("=" * 60)
            print("ðŸ† RESULTADOS:")
            print("=" * 60)
            
            for i, idx in enumerate(indices_ordenados, 1):
                sim_score = similitudes[idx]
                doc = documentos[idx]
                
                # Emoji segÃºn relevancia
                if sim_score >= 0.5:
                    emoji = "ðŸŸ¢"
                elif sim_score >= 0.3:
                    emoji = "ðŸŸ¡"
                else:
                    emoji = "ðŸ”´"
                
                print(f"\n{emoji} Resultado #{i} (similitud: {sim_score:.4f}):")
                print(f"   {doc}")
            
            print("\n" + "=" * 60)
            print("ðŸ’¡ INTERPRETACIÃ“N:")
            print("   ðŸŸ¢ Similitud > 0.5: Muy relevante")
            print("   ðŸŸ¡ Similitud 0.3-0.5: Algo relevante")
            print("   ðŸ”´ Similitud < 0.3: Poco relevante")
    
    buscar_button.on_click(realizar_busqueda)
    
    # Interfaz
    display(VBox([
        widgets.HTML("<h3>ðŸ” Buscador SemÃ¡ntico Interactivo</h3>"),
        widgets.HTML(f"<p style='color: #666;'>Base de datos: {len(documentos)} documentos indexados</p>"),
        busqueda_widget,
        top_k_widget,
        buscar_button,
        resultados_output
    ]))
    
    # Sugerencias de bÃºsqueda
    display(widgets.HTML("""
        <div style='margin-top: 20px; padding: 15px; background-color: #e8f4f8; border-radius: 5px;'>
            <b>ðŸ’¡ Prueba estas bÃºsquedas:</b><br>
            â€¢ "animales domÃ©sticos" â†’ DeberÃ­a encontrar documentos sobre perros y gatos<br>
            â€¢ "programar computadoras" â†’ DeberÃ­a encontrar Python, JavaScript, etc.<br>
            â€¢ "bebidas saludables" â†’ DeberÃ­a encontrar cafÃ©, tÃ©<br>
            â€¢ "inteligencia de mÃ¡quinas" â†’ DeberÃ­a encontrar ML, AI, Deep Learning<br>
            â€¢ "lugares turÃ­sticos" â†’ DeberÃ­a encontrar Torre Eiffel, Coliseo<br><br>
            <b>ðŸŽ¯ Nota:</b> A diferencia de bÃºsquedas tradicionales, aquÃ­ puedes usar <b>sinÃ³nimos</b> 
            y el buscador entenderÃ¡ el significado.
        </div>
    """))
    
else:
    print("â„¹ï¸ Buscador con modelo local no disponible")
    print("   ðŸ’¡ El concepto de bÃºsqueda semÃ¡ntica se aplica igual con APIs")
    print("   âœ… MÃ¡s adelante construiremos buscadores con Gemini")

---
# 3. MODELOS DE EMBEDDINGS (Elige el tuyo)

Ahora que sabes quÃ© son y cÃ³mo usarlos, veamos quÃ© opciones de modelos existen.

## 3.1 Proveedores Principales (APIs en la nube)

### ðŸŒŸ Principales Proveedores de Embeddings

| Proveedor | Modelo Destacado | Dimensiones | Coste | Idiomas | API Gratuita |
|-----------|------------------|-------------|-------|---------|--------------|
| **Google** | text-embedding-004 | 768 | ðŸ†“ GRATIS | 100+ | âœ… SÃ­ (500/dÃ­a) |
| **OpenAI** | text-embedding-3-large | 3072 | $0.13 / 1M tokens | 100+ | âŒ No |
| **OpenAI** | text-embedding-3-small | 1536 | $0.02 / 1M tokens | 100+ | âŒ No |
| **Cohere** | embed-multilingual-v3.0 | 1024 | $0.10 / 1M tokens | 100+ | âœ… SÃ­ (1000/mes) |
| **Anthropic** | (usa viajes API) | - | - | - | âŒ No tiene embeddings |
| **Voyage AI** | voyage-large-2 | 1024 | $0.12 / 1M tokens | InglÃ©s | âš ï¸ Trial |

### ðŸ’° Comparativa de Costes (para 1 millÃ³n de tokens, ~750K palabras)

```
ðŸ†“ Google (text-embedding-004)    : GRATIS (hasta 500 req/dÃ­a)
ðŸ’š OpenAI (small)                  : $0.02
ðŸ’› Cohere                          : $0.10
ðŸ§¡ Voyage AI                       : $0.12
â¤ï¸ OpenAI (large)                  : $0.13
```

### ðŸ† Recomendaciones por Caso de Uso

| Caso de Uso | Modelo Recomendado | Por quÃ© |
|-------------|-------------------|---------|
| **Aprendizaje/Testing** | Google text-embedding-004 | ðŸ†“ Gratis + Excelente calidad |
| **ProducciÃ³n pequeÃ±a** | Google text-embedding-004 | ðŸ†“ Gratis hasta 500 req/dÃ­a |
| **ProducciÃ³n masiva** | OpenAI text-embedding-3-small | Mejor precio/rendimiento |
| **MÃ¡xima precisiÃ³n** | OpenAI text-embedding-3-large | Dimensiones mÃ¡s altas |
| **MultilingÃ¼e** | Cohere embed-multilingual-v3.0 | Especializado en 100+ idiomas |
| **Privacidad total** | Sentence-Transformers (local) | 100% offline, sin APIs |

## 3.2 Modelos Open Source (Locales, sin API)

### ðŸ”“ Ventajas de Modelos Locales

- âœ… **100% GRATIS**: Sin lÃ­mites, sin cuotas
- âœ… **Privacidad Total**: Datos nunca salen de tu ordenador
- âœ… **Sin Internet**: Funciona offline
- âœ… **Sin Latencia**: InstantÃ¡neo (no hay viaje a servidor)
- âŒ **Requiere Hardware**: Necesitas RAM/CPU decente

### ðŸ¤— Top Modelos en HuggingFace

| Modelo | Dimensiones | TamaÃ±o | Idiomas | Mejor para |
|--------|-------------|--------|---------|------------|
| **all-MiniLM-L6-v2** | 384 | 23 MB | InglÃ©s | RÃ¡pido y ligero |
| **paraphrase-multilingual-MiniLM-L12-v2** | 384 | 33 MB | 50+ | MultilingÃ¼e general (el que usamos) |
| **all-mpnet-base-v2** | 768 | 420 MB | InglÃ©s | Mejor calidad inglÃ©s |
| **sentence-t5-base** | 768 | 220 MB | InglÃ©s | Encoders T5 |
| **multilingual-e5-large** | 1024 | 2.2 GB | 100+ | MÃ¡xima calidad multilingÃ¼e |
| **bge-large-en-v1.5** | 1024 | 1.3 GB | InglÃ©s | SOTA inglÃ©s |

### âš¡ Comparativa: Velocidad vs Calidad vs TamaÃ±o

```
        Calidad Alta â†‘
             â”‚
             â”‚   â— multilingual-e5-large (2.2 GB, lento)
             â”‚   
             â”‚       â— bge-large-en-v1.5 (1.3 GB, medio)
             â”‚
             â”‚           â— all-mpnet-base-v2 (420 MB, rÃ¡pido)
             â”‚
             â”‚               â— paraphrase-multilingual-MiniLM-L12-v2 (33 MB) â† RECOMENDADO
             â”‚
             â”‚                   â— all-MiniLM-L6-v2 (23 MB, muy rÃ¡pido)
             â”‚
        Calidad Baja â†“
             â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â†’
                               Velocidad / Ligereza â†’
```

### ðŸŽ¯ Â¿CuÃ¡l elegir?

| SituaciÃ³n | Modelo Recomendado |
|-----------|-------------------|
| **Laptop sin GPU** | all-MiniLM-L6-v2 (23 MB) |
| **EspaÃ±ol/MultilingÃ¼e** | paraphrase-multilingual-MiniLM-L12-v2 â† **Lo que usamos** |
| **Solo inglÃ©s, calidad alta** | all-mpnet-base-v2 |
| **Servidor con GPU, mÃ¡xima calidad** | multilingual-e5-large |
| **Embedding de cÃ³digo** | microsoft/codebert-base |

## 3.3 Tabla Comparativa: TamaÃ±o, Idiomas, Coste

### ðŸ“Š Comparativa Completa: APIs vs Local

| Modelo | Tipo | Dimensiones | Idiomas | Coste API | Local | Calidad* |
|--------|------|-------------|---------|-----------|-------|----------|
| **Google text-embedding-004** | API | 768 | 100+ | ðŸ†“ Gratis | âŒ | â­â­â­â­â­ |
| **OpenAI text-embedding-3-large** | API | 3072 | 100+ | $0.13/1M | âŒ | â­â­â­â­â­ |
| **OpenAI text-embedding-3-small** | API | 1536 | 100+ | $0.02/1M | âŒ | â­â­â­â­ |
| **Cohere embed-multilingual-v3.0** | API | 1024 | 100+ | $0.10/1M | âŒ | â­â­â­â­â­ |
| **multilingual-e5-large** | Local | 1024 | 100+ | ðŸ†“ Gratis | âœ… 2.2 GB | â­â­â­â­â­ |
| **all-mpnet-base-v2** | Local | 768 | InglÃ©s | ðŸ†“ Gratis | âœ… 420 MB | â­â­â­â­ |
| **paraphrase-multilingual-MiniLM-L12-v2** | Local | 384 | 50+ | ðŸ†“ Gratis | âœ… 33 MB | â­â­â­â­ |
| **all-MiniLM-L6-v2** | Local | 384 | InglÃ©s | ðŸ†“ Gratis | âœ… 23 MB | â­â­â­ |

*Calidad aproximada en benchmarks estÃ¡ndar (MTEB)

### ðŸ’¡ Reglas PrÃ¡cticas

1. **Â¿Tienes API key de Gemini?** â†’ Usa **Google text-embedding-004** (gratis + excelente)
2. **Â¿No tienes API key o valoras privacidad?** â†’ Usa **paraphrase-multilingual-MiniLM-L12-v2** (local)
3. **Â¿Solo inglÃ©s + quieres mÃ¡xima calidad local?** â†’ Usa **all-mpnet-base-v2**
4. **Â¿ProducciÃ³n a gran escala?** â†’ Usa **OpenAI text-embedding-3-small** (mejor precio/rendimiento)
5. **Â¿Laptop antiguo/limitado?** â†’ Usa **all-MiniLM-L6-v2** (el mÃ¡s ligero)

### ðŸ”¬ Benchmarks Reales (MTEB - Massive Text Embedding Benchmark)

Estos son los scores reales de los modelos (0-100, mayor es mejor):

| Modelo | MTEB Score | Ranking |
|--------|------------|---------|
| OpenAI text-embedding-3-large | 64.6 | ðŸ¥‡ #1 |
| Cohere embed-multilingual-v3.0 | 62.8 | ðŸ¥ˆ #2 |
| Google text-embedding-004 | ~61.0 | ðŸ¥‰ #3 |
| multilingual-e5-large | 57.3 | #7 |
| all-mpnet-base-v2 | 57.8 | #5 |
| paraphrase-multilingual-MiniLM-L12-v2 | 52.1 | #45 |
| all-MiniLM-L6-v2 | 48.2 | #78 |

**ðŸ’¡ Nota:** Los modelos API (OpenAI, Google, Cohere) suelen superar a los locales en calidad, pero los locales son gratis e ilimitados.


## 3.4 Ejercicio: Â¿QuÃ© modelo para cada caso?

### ðŸŽ¯ EJERCICIO PRÃCTICO

Lee los siguientes 5 casos de uso y decide quÃ© modelo de embeddings usarÃ­as. Hay mÃºltiples respuestas correctas, pero intenta justificar tu elecciÃ³n.

---

#### **Caso 1: Startup con MVP (Minimum Viable Product)**
- Presupuesto muy limitado ($0 idealmente)
- BÃºsqueda semÃ¡ntica en 500 documentos
- Solo espaÃ±ol
- Usuarios: ~100/dÃ­a

**ðŸ¤” Â¿QuÃ© modelo elegirÃ­as y por quÃ©?**


In [ ]:
# ðŸŽ¯ SoluciÃ³n del Ejercicio - Caso 1

print("ðŸ’¡ SOLUCIÃ“N RECOMENDADA PARA CASO 1:")
print("=" * 60)
print("ðŸ† Modelo recomendado: Google text-embedding-004")
print("\nâœ… Razones:")
print("   1. ðŸ†“ Completamente GRATIS (0 presupuesto)")
print("   2. âœ… Soporta espaÃ±ol perfectamente (100+ idiomas)")
print("   3. â­ Excelente calidad (top 3 mundial)")
print("   4. ðŸ“Š 500 req/dÃ­a es suficiente para 100 usuarios")
print("   5. ðŸš€ RÃ¡pido (sin instalar nada)")
print("\nâš ï¸ Alternativa si valoran privacidad:")
print("   ðŸ”“ paraphrase-multilingual-MiniLM-L12-v2 (local)")
print("   - 100% gratis e ilimitado")
print("   - Funciona offline")
print("   - Menor calidad pero suficiente para MVP")
print("=" * 60)

---

#### **Caso 2: Empresa Grande con Millones de Documentos**
- Presupuesto: $5,000/mes
- BÃºsqueda semÃ¡ntica en 10 millones de documentos
- MultilingÃ¼e (inglÃ©s, espaÃ±ol, francÃ©s, alemÃ¡n)
- Usuarios: ~50,000/dÃ­a
- Requisito: MÃ¡xima precisiÃ³n

**ðŸ¤” Â¿QuÃ© modelo elegirÃ­as y por quÃ©?**


In [ ]:
# ðŸŽ¯ SoluciÃ³n del Ejercicio - Caso 2

print("ðŸ’¡ SOLUCIÃ“N RECOMENDADA PARA CASO 2:")
print("=" * 60)
print("ðŸ† Modelo recomendado: OpenAI text-embedding-3-small")
print("\nâœ… Razones:")
print("   1. ðŸ’° Muy econÃ³mico: $0.02 / 1M tokens")
print("      - 10M docs Ã— 500 tokens/doc = 5,000M tokens")
print("      - Coste one-time indexado: 5,000 Ã— $0.02 = $100")
print("      - BÃºsquedas: 50K/dÃ­a Ã— 30 dÃ­as Ã— 100 tokens Ã— $0.02/1M = $3/mes")
print("   2. â­ Calidad excelente (top 5 mundial)")
print("   3. ðŸŒ MultilingÃ¼e (100+ idiomas)")
print("   4. ðŸš€ Muy rÃ¡pido y escalable")
print("   5. ðŸ“ 1536 dimensiones (buen balance)")
print("\nâš ï¸ Alternativa si necesitan MÃXIMA precisiÃ³n:")
print("   ðŸ¥‡ OpenAI text-embedding-3-large")
print("   - 3072 dimensiones (doble capacidad)")
print("   - $0.13 / 1M tokens (6.5x mÃ¡s caro, pero dentro de presupuesto)")
print("\nâš ï¸ NO recomendado en este caso:")
print("   âŒ Google text-embedding-004: LÃ­mite 500 req/dÃ­a insuficiente")
print("   âŒ Modelos locales: Requieren infraestructura cara para 10M docs")
print("=" * 60)

---

#### **Caso 3: App MÃ©dica con Datos Sensibles**
- Requisito CRÃTICO: Privacidad absoluta (HIPAA compliance)
- Datos mÃ©dicos confidenciales
- NO pueden enviar datos a servicios externos
- 10,000 documentos mÃ©dicos
- Solo inglÃ©s
- Presupuesto: $2,000 para servidor

**ðŸ¤” Â¿QuÃ© modelo elegirÃ­as y por quÃ©?**


In [ ]:
# ðŸŽ¯ SoluciÃ³n del Ejercicio - Caso 3

print("ðŸ’¡ SOLUCIÃ“N RECOMENDADA PARA CASO 3:")
print("=" * 60)
print("ðŸ† Modelo recomendado: all-mpnet-base-v2 (local)")
print("\nâœ… Razones:")
print("   1. ðŸ”’ 100% PRIVADO - Datos nunca salen del servidor")
print("   2. âœ… HIPAA compliant (sin APIs externas)")
print("   3. â­ Mejor modelo local para inglÃ©s (57.8 MTEB)")
print("   4. ðŸ’¾ Solo 420 MB - Cabe en cualquier servidor")
print("   5. ðŸ†“ GRATIS - Sin costes API recurrentes")
print("   6. âš¡ RÃ¡pido en CPU (no requiere GPU)")
print("\nâš ï¸ Alternativa si tienen GPU y necesitan MÃS calidad:")
print("   ðŸ¥‡ multilingual-e5-large")
print("   - AÃºn mejor calidad (57.3 MTEB)")
print("   - 2.2 GB pero servidor con GPU puede manejarlo")
print("\nâŒ NO recomendado:")
print("   âŒ CUALQUIER API (Google, OpenAI, Cohere)")
print("   - ViolarÃ­a HIPAA (datos salen a servidores externos)")
print("   - Riesgo legal y Ã©tico con datos mÃ©dicos")
print("=" * 60)

---

#### **Caso 4: Estudiante Aprendiendo IA**
- Presupuesto: $0 (estudiante)
- Laptop normal (8 GB RAM, sin GPU)
- Proyecto personal: Chatbot con 100 preguntas FAQ
- EspaÃ±ol
- Solo para demostrar en clase

**ðŸ¤” Â¿QuÃ© modelo elegirÃ­as y por quÃ©?**


In [ ]:
# ðŸŽ¯ SoluciÃ³n del Ejercicio - Caso 4

print("ðŸ’¡ SOLUCIÃ“N RECOMENDADA PARA CASO 4:")
print("=" * 60)
print("ðŸ† OPCIÃ“N 1 (Recomendada): Google text-embedding-004")
print("\nâœ… Razones:")
print("   1. ðŸ†“ Completamente GRATIS")
print("   2. ðŸš€ No consume recursos de tu laptop")
print("   3. â­ Calidad excelente (impresiona al profesor)")
print("   4. ðŸ“š 100 FAQs = ~10 requests (muy por debajo del lÃ­mite)")
print("   5. âœ… Perfecto para demostrar conceptos")
print("\nðŸ† OPCIÃ“N 2 (Alternativa sin internet):")
print("   ðŸ”“ all-MiniLM-L6-v2 (23 MB, local)")
print("\nâœ… Razones:")
print("   1. ðŸ†“ Gratis e ilimitado")
print("   2. ðŸ’¾ Solo 23 MB (corre en cualquier laptop)")
print("   3. âš¡ Muy rÃ¡pido incluso sin GPU")
print("   4. ðŸ“´ Funciona sin internet")
print("   5. âš ï¸ Solo inglÃ©s, pero puedes hacer traducciÃ³n previa")
print("\nðŸ’¡ ESTRATEGIA COMBINADA (lo mejor):")
print("   - Desarrollo/testing: Usa all-MiniLM-L6-v2 (local, rÃ¡pido)")
print("   - Demo en clase: Cambia a Google text-embedding-004 (impresiona)")
print("=" * 60)

---

#### **Caso 5: Buscador de CÃ³digo Interno**
- Empresa de software
- Buscar cÃ³digo similar en repos privados
- ~50,000 archivos de cÃ³digo (Python, JavaScript, Java)
- Requisito: Privacidad (cÃ³digo propietario)
- Presupuesto: $500/mes
- Infraestructura: Servidor con GPU Tesla T4

**ðŸ¤” Â¿QuÃ© modelo elegirÃ­as y por quÃ©?**


In [ ]:
# ðŸŽ¯ SoluciÃ³n del Ejercicio - Caso 5

print("ðŸ’¡ SOLUCIÃ“N RECOMENDADA PARA CASO 5:")
print("=" * 60)
print("ðŸ† Modelo recomendado: microsoft/codebert-base (local)")
print("\nâœ… Razones:")
print("   1. ðŸ’» Especializado en CÃ“DIGO (entrenado en GitHub)")
print("   2. ðŸ”’ 100% privado (cÃ³digo no sale de la empresa)")
print("   3. ðŸ†“ Gratis - Ahorra el presupuesto")
print("   4. ðŸŽ® GPU Tesla T4 es PERFECTA para este modelo")
print("   5. âš¡ Muy rÃ¡pido con GPU (puede indexar 50K archivos rÃ¡pido)")
print("   6. ðŸŒ Entiende sintaxis de Python, JS, Java, etc.")
print("\nðŸ“Š Especificaciones:")
print("   - Dimensiones: 768")
print("   - TamaÃ±o: 500 MB")
print("   - Lenguajes: Python, Java, JavaScript, PHP, Ruby, Go")
print("\nâš ï¸ Alternativa si quieren mejor calidad general:")
print("   ðŸ¥‡ multilingual-e5-large")
print("   - TambiÃ©n privado y gratis")
print("   - Mejor en texto natural (comentarios)")
print("   - Pero CodeBERT es mejor para sintaxis de cÃ³digo")
print("\nâŒ NO usar APIs en este caso:")
print("   - CÃ³digo propietario no deberÃ­a ir a OpenAI/Google")
print("   - Riesgo de fugas de IP (Intellectual Property)")
print("=" * 60)

print("\n\nðŸ“š CÃ“MO CARGAR CODEBERT:")
print("-" * 60)
print("from sentence_transformers import SentenceTransformer")
print("model = SentenceTransformer('microsoft/codebert-base')")
print("")
print("# Ejemplo de uso")
print("codigo1 = 'def factorial(n): return 1 if n == 0 else n * factorial(n-1)'")
print("codigo2 = 'def fib(n): return n if n <= 1 else fib(n-1) + fib(n-2)'")
print("embeddings = model.encode([codigo1, codigo2])")
print("-" * 60)

### ðŸŽ“ Resumen del Ejercicio

**Lecciones clave aprendidas:**

1. **No hay "un mejor modelo para todo"** - Depende del caso de uso
2. **Considera SIEMPRE:**
   - ðŸ’° Presupuesto
   - ðŸ”’ Requisitos de privacidad/seguridad
   - ðŸ“Š Volumen de datos
   - ðŸŒ Idiomas necesarios
   - âš¡ Recursos disponibles (GPU, CPU, RAM)
3. **Para empezar/aprender:** Google text-embedding-004 (gratis + excelente)
4. **Para producciÃ³n:** OpenAI text-embedding-3-small (precio/rendimiento)
5. **Para privacidad:** Modelos locales (Sentence-Transformers)
6. **Para cÃ³digo:** microsoft/codebert-base (especializado)

---
# 4. ðŸŽ« TOKENS: La moneda del mundo LLM

Ahora que entiendes embeddings, vamos a hablar de **tokens** - la forma en que se mide y cobra el uso de APIs.

## 4.1 Â¿QuÃ© es un token? (No es palabra ni carÃ¡cter)

### ðŸ¤” El Problema: Â¿CÃ³mo "cortar" el texto?

Cuando un LLM procesa texto, necesita dividirlo en pedazos. Hay 3 opciones:

#### âŒ OpciÃ³n 1: Por caracteres
```
Texto: "Hola mundo"
Caracteres: H-o-l-a- -m-u-n-d-o (10 caracteres)
Problema: Demasiado fragmentado, "Hola" pierde significado
```

#### âŒ OpciÃ³n 2: Por palabras completas
```
Texto: "anticonstitucionalmente"
Problema: Â¿Es 1 palabra o deberÃ­a ser sub-partes?
Vocabulario infinito en espaÃ±ol/inglÃ©s
```

#### âœ… OpciÃ³n 3: Por TOKENS (subpalabras)
```
Texto: "anticonstitucionalmente"
Tokens: "anti" + "constitu" + "cional" + "mente"
Ventaja: Balance entre granularidad y vocabulario finito
```

---

### ðŸ§© DefiniciÃ³n de Token

Un **token** es una **unidad bÃ¡sica de texto** para los LLMs, puede ser:

- Una palabra completa: `"perro"` â†’ 1 token
- Parte de una palabra: `"anti" + "constitu" + "cional" + "mente"` â†’ 4 tokens
- Un signo de puntuaciÃ³n: `","` â†’ 1 token
- Un espacio + palabra corta: `" el"` â†’ 1 token

---

### ðŸ“Š Ejemplos PrÃ¡cticos

| Texto | Tokens (aprox) | ExplicaciÃ³n |
|-------|----------------|-------------|
| `"Hola"` | 1 | Palabra corta comÃºn = 1 token |
| `"Hola mundo"` | 2 | `"Hola"` + `" mundo"` |
| `"anticonstitucionalmente"` | 4-5 | Palabra larga se fragmenta |
| `"I love AI"` | 3 | `"I"` + `" love"` + `" AI"` |
| `"    "` (4 espacios) | 1 | Espacios se agrupan |
| `"GPT-4"` | 2-3 | `"GPT"` + `"-"` + `"4"` |
| `"ðŸ˜Š"` | 1-2 | Emojis suelen ser 1-2 tokens |
| `"cÃ³digo Python"` | 3 | `"cÃ³digo"` (acento) + `" Python"` |

---

### ðŸ”¬ Â¿CÃ³mo se crean los tokens?

Los LLMs usan un proceso llamado **TokenizaciÃ³n** con algoritmos como **BPE (Byte Pair Encoding)** o **WordPiece**.

**Paso a paso:**

1. **Entrenar en texto masivo:**
   - Analizar millones de libros/internet
   - Identificar combinaciones de caracteres mÃ¡s frecuentes

2. **Crear vocabulario finito:**
   - GPT-4: ~100,000 tokens en su vocabulario
   - Cada token tiene un ID numÃ©rico Ãºnico

3. **Tokenizar texto nuevo:**
   - Dividir usando las reglas aprendidas
   - Convertir cada token a su ID numÃ©rico

**Ejemplo visual:**

```
Texto original: "TokenizaciÃ³n es interesante"

Paso 1 - Tokenizar:
["Token", "izaciÃ³n", " es", " interesante"]

Paso 2 - Convertir a IDs:
[49502, 13821, 655, 23483]

Paso 3 - Procesar con LLM:
[49502, 13821, 655, 23483] â†’ Modelo â†’ Respuesta
```

---

### ðŸ’¡ Conceptos Clave

- **1 token â‰ˆ 4 caracteres** en inglÃ©s
- **1 token â‰ˆ 0.75 palabras** en inglÃ©s
- **EspaÃ±ol/idiomas con acentos:** MÃ¡s tokens que inglÃ©s (acentos se fragmentan)
- **CÃ³digo:** MÃ¡s tokens que texto natural (sintaxis se fragmenta)
- **Tokens = CÃ³mo se MIDE el uso de APIs** (no palabras ni caracteres)

---

### ðŸŽ¯ Â¿Por quÃ© importa?

1. **ðŸ’° COSTES:** Las APIs cobran por token, Â¡no por palabra!
2. **ðŸ“ LÃMITES:** Los modelos tienen lÃ­mite de tokens (contexto)
3. **âš¡ VELOCIDAD:** MÃ¡s tokens = MÃ¡s tiempo de procesamiento

**Ejemplo de diferencia de coste:**

```
GPT-4: $0.03 / 1,000 tokens

Texto: "ReuniÃ³n sobre estrategia de marketing digital" (7 palabras)

TokenizaciÃ³n:
- InglÃ©s: "Meeting about digital marketing strategy" â†’ 5 tokens
- EspaÃ±ol: "ReuniÃ³n sobre estrategia de marketing digital" â†’ 9 tokens

Coste:
- InglÃ©s: $0.00015
- EspaÃ±ol: $0.00027 (80% mÃ¡s caro!)
```

ðŸ’¡ **Por eso es importante optimizar tokens, Â¡lo veremos en la secciÃ³n 6!**

## 4.2 Tipos de Tokens: Input, Output, Razonamiento

Las APIs modernas distinguen entre diferentes tipos de tokens, y **cada tipo puede tener un precio diferente**.

### ðŸ“Š Tipos de Tokens

| Tipo | QuÃ© es | Ejemplo | Precio tÃ­pico |
|------|--------|---------|---------------|
| **Input Tokens** | Tokens que TÃš envÃ­as (prompt) | Tu pregunta, contexto, documentos | MÃ¡s barato |
| **Output Tokens** | Tokens que el modelo GENERA (respuesta) | La respuesta del modelo | MÃ¡s caro (2-3x) |
| **Reasoning Tokens** | Tokens de "pensamiento" interno (solo modelos o1) | Razonamiento paso a paso oculto | Muy caro |
| **Cached Tokens** | Tokens reutilizados de contexto previo | Prompt que ya enviaste antes | 90% descuento |

---

### ðŸ’° Ejemplo Real: Precios de GPT-4

| Modelo | Input Tokens | Output Tokens | Razonamiento |
|--------|--------------|---------------|--------------|
| **GPT-4o** | $2.50 / 1M | $10.00 / 1M | - |
| **GPT-4o-mini** | $0.15 / 1M | $0.60 / 1M | - |
| **o1** | $15.00 / 1M | $60.00 / 1M | $60.00 / 1M |
| **o1-mini** | $3.00 / 1M | $12.00 / 1M | $12.00 / 1M |

**ðŸ’¡ Observa:** Output tokens cuestan **4x mÃ¡s** que input tokens en la mayorÃ­a de modelos.

---

### ðŸ§  Tokens de Razonamiento (Reasoning Tokens)

Los nuevos modelos **o1** de OpenAI tienen un tipo especial de tokens:

```
Usuario: "Â¿CuÃ¡ntos Rs hay en 'strawberry'?"

â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”
â”‚ ðŸ§  REASONING TOKENS (ocultos al usuario) â”‚
â”œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¤
â”‚ Paso 1: Voy a contar letra por letra    â”‚
â”‚ Paso 2: s-t-r-a-w-b-e-r-r-y              â”‚
â”‚ Paso 3: Identifico: r en posiciÃ³n 3     â”‚
â”‚ Paso 4: Identifico: r en posiciÃ³n 9     â”‚
â”‚ Paso 5: Identifico: r en posiciÃ³n 10    â”‚
â”‚ ConclusiÃ³n: 3 letras R                  â”‚
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜
         â†“
â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”
â”‚ ðŸ’¬ OUTPUT TOKENS (visible al usuario)    â”‚
â”œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¤
â”‚ Hay 3 letras 'R' en la palabra          â”‚
â”‚ 'strawberry': posiciones 3, 9 y 10.     â”‚
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜
```

**ðŸ’° Costes:**
- Input: "Â¿CuÃ¡ntos Rs hay en 'strawberry'?" â†’ ~10 tokens Ã— $15/1M = $0.00015
- Reasoning: ~500 tokens de razonamiento oculto Ã— $60/1M = $0.03
- Output: "Hay 3 letras R..." â†’ ~15 tokens Ã— $60/1M = $0.0009
- **TOTAL: $0.031** (la mayor parte es razonamiento interno)

---

### ðŸ’¾ Cached Tokens (CachÃ© de Prompts)

Algunos proveedores (OpenAI, Anthropic) permiten **reutilizar** partes del prompt que no cambian.

**Ejemplo: Chatbot con manual de producto**

```
ConversaciÃ³n 1:
â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”
â”‚ SYSTEM PROMPT (10,000 tokens)           â”‚
â”‚ [Manual completo del producto...]       â”‚
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜
USER: "Â¿CÃ³mo se carga?" (5 tokens)

Coste:
- Input: 10,005 tokens Ã— $2.50/1M = $0.025
- Output: ~50 tokens Ã— $10/1M = $0.0005
- TOTAL: $0.0255

---

ConversaciÃ³n 2 (mismo usuario, 5 min despues):
â”Œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”
â”‚ SYSTEM PROMPT (10,000 tokens) â† CACHED  â”‚
â”‚ [Manual completo del producto...]       â”‚
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜
USER: "Â¿Y la baterÃ­a dura cuÃ¡nto?" (7 tokens)

Coste con cachÃ©:
- Cached input: 10,000 tokens Ã— $0.25/1M = $0.0025 (90% descuento!)
- New input: 7 tokens Ã— $2.50/1M = $0.000018
- Output: ~60 tokens Ã— $10/1M = $0.0006
- TOTAL: $0.00312 (8x mÃ¡s barato!)
```

**ðŸ’¡ Regla:** Si envÃ­as el **mismo contexto largo** repetidamente (ej: manual, documentos), usa cachÃ©.

---

### ðŸŽ¯ Estrategia de OptimizaciÃ³n por Tipo de Token

| Objetivo | Estrategia |
|----------|------------|
| **Reducir Input Tokens** | - Acortar prompts innecesarios<br>- Eliminar ejemplos redundantes<br>- Resumir documentos largos |
| **Reducir Output Tokens** | - Usar `max_tokens` para limitar respuesta<br>- Pedir respuestas concisas<br>- Usar formatos estructurados (JSON) |
| **Aprovechar CachÃ©** | - Mantener system prompts constantes<br>- Reutilizar contexto entre llamadas<br>- Usar sesiones persistentes |
| **Evitar Reasoning Tokens** | - Usar o1 solo cuando necesites razonamiento complejo<br>- Para tareas simples, usa GPT-4o-mini |


## 4.3 Contador de Tokens Interactivo

Vamos a crear un contador de tokens para que veas EXACTAMENTE cuÃ¡ntos tokens consume tu texto.

Usaremos **tiktoken**, la librerÃ­a oficial de OpenAI.

In [ ]:
# ðŸŽ« Contador de Tokens Interactivo

if TIKTOKEN_AVAILABLE:
    
    print("ðŸ”„ Cargando tokenizador de OpenAI...")
    
    # Tokenizadores disponibles
    # - cl100k_base: GPT-4, GPT-3.5-turbo, text-embedding-ada-002
    # - p50k_base: Code models (Codex)
    # - r50k_base: GPT-3 (davinci, curie, etc.)
    
    encoding = tiktoken.get_encoding("cl100k_base")  # GPT-4 / GPT-3.5
    print("âœ… Tokenizador cargado (cl100k_base - GPT-4)\n")
    
    # Widgets
    input_texto = Textarea(
        placeholder='Escribe o pega tu texto aquÃ­...\n\nPuedes escribir:\n- Prompts\n- CÃ³digo\n- Documentos\n- Cualquier cosa',
        description='Texto:',
        layout=Layout(width='800px', height='200px')
    )
    
    contar_button = Button(
        description='ðŸŽ« Contar Tokens',
        button_style='primary',
        layout=Layout(width='200px')
    )
    
    output_tokens = Output()
    
    def contar_tokens(b):
        with output_tokens:
            output_tokens.clear_output()
            
            if not input_texto.value.strip():
                print("âš ï¸ Por favor, escribe algo primero")
                return
            
            texto = input_texto.value
            
            # Tokenizar
            tokens = encoding.encode(texto)
            num_tokens = len(tokens)
            
            # Decodificar cada token para visualizarlo
            tokens_decodificados = [encoding.decode([token]) for token in tokens]
            
            print("=" * 60)
            print("ðŸ“Š ANÃLISIS DE TOKENS")
            print("=" * 60)
            
            # EstadÃ­sticas bÃ¡sicas
            num_caracteres = len(texto)
            num_palabras = len(texto.split())
            
            print(f"\nðŸ“ EstadÃ­sticas:")
            print(f"   - Caracteres: {num_caracteres}")
            print(f"   - Palabras: {num_palabras}")
            print(f"   - TOKENS: {num_tokens}")
            print(f"\nðŸ“Š Ratios:")
            print(f"   - Tokens/Palabra: {num_tokens/num_palabras:.2f}")
            print(f"   - Caracteres/Token: {num_caracteres/num_tokens:.2f}")
            
            # EstimaciÃ³n de costes
            print(f"\nðŸ’° ESTIMACIÃ“N DE COSTES (GPT-4o):")
            print(f"   Como Input:  ${(num_tokens / 1_000_000) * 2.50:.6f}")
            print(f"   Como Output: ${(num_tokens / 1_000_000) * 10.00:.6f}")
            
            print(f"\nðŸ’° ESTIMACIÃ“N DE COSTES (GPT-4o-mini):")
            print(f"   Como Input:  ${(num_tokens / 1_000_000) * 0.15:.6f}")
            print(f"   Como Output: ${(num_tokens / 1_000_000) * 0.60:.6f}")
            
            # Mostrar primeros 50 tokens
            print(f"\nðŸ” PRIMEROS 50 TOKENS (de {num_tokens}):")
            print("=" * 60)
            
            for i, token_text in enumerate(tokens_decodificados[:50], 1):
                # Reemplazar saltos de lÃ­nea y tabs para visualizaciÃ³n
                token_visual = token_text.replace('\n', '\\n').replace('\t', '\\t').replace(' ', 'Â·')
                print(f"   {i:2}. [{tokens[i-1]:5}] '{token_visual}'")
            
            if num_tokens > 50:
                print(f"\n   ... (omitidos {num_tokens - 50} tokens mÃ¡s)")
            
            print("=" * 60)
            
            print("\nðŸ’¡ OBSERVACIONES:")
            if num_tokens / num_palabras > 1.5:
                print("   âš ï¸ Tu texto usa MUCHOS tokens por palabra")
                print("      Posibles razones:")
                print("      - Idioma con acentos (espaÃ±ol, francÃ©s, etc.)")
                print("      - Palabras tÃ©cnicas/especÃ­ficas")
                print("      - CÃ³digo o sintaxis especial")
            elif num_tokens / num_palabras < 1.2:
                print("   âœ… Tu texto es eficiente en tokens")
                print("      - Palabras cortas y comunes")
                print("      - Buena proporciÃ³n palabras/tokens")
            else:
                print("   âœ… ProporciÃ³n normal de tokens")
    
    contar_button.on_click(contar_tokens)
    
    # Interfaz
    display(VBox([
        widgets.HTML("<h3>ðŸŽ« Contador de Tokens (GPT-4 Tokenizer)</h3>"),
        widgets.HTML("<p style='color: #666;'>Escribe cualquier texto y descubre cuÃ¡ntos tokens consume:</p>"),
        input_texto,
        contar_button,
        output_tokens
    ]))
    
    # Ejemplos para probar
    display(widgets.HTML("""
        <div style='margin-top: 20px; padding: 15px; background-color: #fff3cd; border-radius: 5px;'>
            <b>ðŸ’¡ Prueba estos textos para comparar:</b><br><br>
            <b>Texto 1 (inglÃ©s, eficiente):</b><br>
            <code>Hello world</code> â†’ ~2 tokens<br><br>
            <b>Texto 2 (espaÃ±ol, mÃ¡s tokens):</b><br>
            <code>Hola mundo</code> â†’ ~3 tokens<br><br>
            <b>Texto 3 (cÃ³digo Python):</b><br>
            <code>def factorial(n): return 1 if n == 0 else n * factorial(n-1)</code><br><br>
            <b>Texto 4 (palabra larga):</b><br>
            <code>anticonstitucionalmente</code> â†’ ~6-7 tokens
        </div>
    """))
    
else:
    print("âš ï¸ tiktoken no estÃ¡ instalado")
    print("   SoluciÃ³n: pip install tiktoken")

## 4.4 Regla PrÃ¡ctica: 1 token â‰ˆ 0.75 palabras

Para hacer estimaciones rÃ¡pidas sin tokenizar, puedes usar esta regla:

### ðŸ“ Reglas de Oro para EstimaciÃ³n RÃ¡pida

| Idioma/Tipo | Ratio Tokens/Palabra | Ejemplo |
|-------------|----------------------|---------|
| **InglÃ©s** | 0.75 tokens/palabra | 100 palabras â‰ˆ 75 tokens |
| **EspaÃ±ol** | 1.3 tokens/palabra | 100 palabras â‰ˆ 130 tokens |
| **CÃ³digo** | 1.5-2 tokens/palabra | 100 tokens de cÃ³digo â‰ˆ 150 tokens |
| **JSON/XML** | 2-3 tokens/palabra | Muchos sÃ­mbolos = muchos tokens |

### ðŸŽ¯ FÃ³rmulas RÃ¡pidas

```python
# EstimaciÃ³n rÃ¡pida para INGLÃ‰S
tokens_estimados = palabras Ã— 1.3

# EstimaciÃ³n rÃ¡pida para ESPAÃ‘OL
tokens_estimados = palabras Ã— 1.6

# EstimaciÃ³n rÃ¡pida para CÃ“DIGO
tokens_estimados = palabras Ã— 2
```

### ðŸ’° CÃ¡lculo RÃ¡pido de Costes

**Ejemplo: ArtÃ­culo de 1000 palabras en espaÃ±ol**

```
1. Estimar tokens:
   1000 palabras Ã— 1.6 = 1,600 tokens

2. Calcular coste (GPT-4o como input):
   1,600 tokens / 1,000,000 Ã— $2.50 = $0.004

3. Si el modelo responde con 500 palabras:
   500 palabras Ã— 1.6 = 800 tokens
   800 tokens / 1,000,000 Ã— $10.00 = $0.008

4. TOTAL: $0.012 (un cÃ©ntimo y pico)
```

### ðŸ“Š Tabla de Referencia RÃ¡pida

| Texto | Palabras | Tokens (estimado) | Coste GPT-4o (input) |
|-------|----------|-------------------|----------------------|
| Tweet | 20 | 30 | $0.000075 |
| Email corto | 100 | 150 | $0.000375 |
| PÃ¡gina A4 | 500 | 800 | $0.002 |
| ArtÃ­culo blog | 2,000 | 3,200 | $0.008 |
| Libro capÃ­tulo | 10,000 | 16,000 | $0.04 |
| Libro completo | 80,000 | 128,000 | $0.32 |

ðŸ’¡ **Nota:** Estos son valores aproximados. Para precisiÃ³n, usa siempre un tokenizador real.


---
# 5. ðŸ’° PRECIOS Y CALCULADORA DE COSTES

Ahora que sabes contar tokens, aprenderÃ¡s a calcular costes reales de APIs.

parativos 2026

A continuaciÃ³n, una tabla con los precios reales actualizados de los modelos mÃ¡s populares (Marzo 2026):

| Modelo | Input ($/1M tokens) | Output ($/1M tokens) | Ratio Output/Input |
|--------|--------------------|-----------------------|--------------------|
| **GPT-4o** | $2.50 | $10.00 | 4x |
| **GPT-4o-mini** | $0.15 | $0.60 | 4x |
| **GPT-4 Turbo** | $10.00 | $30.00 | 3x |
| **o1** | $15.00 | $60.00 | 4x |
| **o1-mini** | $3.00 | $12.00 | 4x |
| **Claude 4 Opus** | $15.00 | $75.00 | 5x |
| **Claude 4 Sonnet** | $3.00 | $15.00 | 5x |
| **Claude 4 Haiku** | $0.25 | $1.25 | 5x |
| **Gemini 3 Pro** | $1.25 | $5.00 | 4x |
| **Gemini 3 Flash** | $0.075 | $0.30 | 4x |
| **Gemini 2.5 Flash Lite** | ðŸ†“ GRATIS | ðŸ†“ GRATIS | - |

### ðŸ’¡ Observaciones Clave

1. **Output siempre cuesta MÃS** (3-5x mÃ¡s caro que input)
2. **Modelos "mini/small/haiku" son 10-50x mÃ¡s baratos** que los grandes
3. **Gemini Flash es el MÃS BARATO** (casi gratis)
4. **Claude Opus es el MÃS CARO** (mÃ¡xima calidad, mÃ¡ximo precio)

---

### ðŸŽ¯ Â¿QuÃ© Modelo Elegir SegÃºn Presupuesto?

| Presupuesto | Modelo Recomendado | Coste (1M tokens) |
|-------------|-------------------|-------------------|
| **$0 (Gratis)** | Gemini 2.5 Flash Lite | ðŸ†“ |
| **< $1** | Gemini 3 Flash | $0.075 input |
| **$1-5** | GPT-4o-mini / Claude 4 Haiku | $0.15-0.25 input |
| **$5-15** | GPT-4o / Gemini 3 Pro / Claude 4 Sonnet | $1.25-3 input |
| **$15+** | o1 / Claude 4 Opus / GPT-4 Turbo | $10-15 input |

---

### ðŸ“ FÃ³rmula Universal para Calcular Costes

```python
coste_total = (input_tokens / 1_000_000 Ã— precio_input) + 
              (output_tokens / 1_000_000 Ã— precio_output)
```

**Ejemplo numÃ©rico:**
```
Modelo: GPT-4o
Input: 2,000 tokens
Output: 500 tokens

CÃ¡lculo:
= (2,000 / 1,000,000 Ã— $2.50) + (500 / 1,000,000 Ã— $10.00)
= (0.002 Ã— $2.50) + (0.0005 Ã— $10.00)
= $0.005 + $0.005
= $0.01 (1 cÃ©ntimo por llamada)
```

## 5.2 Ejercicio: CÃ¡lculo Manual

### ðŸŽ¯ EJERCICIO PRÃCTICO

Tienes una **aplicaciÃ³n de resÃºmenes** que:
- Recibe documentos de 3,000 palabras (input)
- Genera resÃºmenes de 300 palabras (output)
- Usa el modelo **GPT-4o-mini**

**Precios de GPT-4o-mini:**
- Input: $0.15 / 1M tokens
- Output: $0.60 / 1M tokens

**EstimaciÃ³n de tokens:**
- 1 palabra â‰ˆ 1.3 tokens (inglÃ©s/espaÃ±ol promedio)

---

### â“ PREGUNTAS:

**1. Â¿CuÃ¡ntos tokens de input por documento?**

**2. Â¿CuÃ¡ntos tokens de output por resumen?**

**3. Â¿CuÃ¡nto cuesta procesar 1 documento?**

**4. Â¿CuÃ¡nto costarÃ­a procesar 1,000 documentos?**

**5. Â¿CuÃ¡nto costarÃ­a procesar 10,000 documentos al mes?**

---

### ðŸ’­ **PiÃ©nsalo antes de ver la soluciÃ³n...**

(DesplÃ¡zate hacia abajo para ver las respuestas)

<br><br><br><br><br>

In [ ]:
# ðŸŽ¯ SOLUCIÃ“N DEL EJERCICIO

print("ðŸ’¡ SOLUCIÃ“N PASO A PASO")
print("=" * 60)

# Datos del problema
palabras_input = 3000
palabras_output = 300
ratio_tokens = 1.3  # 1 palabra â‰ˆ 1.3 tokens

precio_input_por_millon = 0.15
precio_output_por_millon = 0.60

print("\nðŸ“‹ DATOS:")
print(f"   - Documento: {palabras_input} palabras")
print(f"   - Resumen: {palabras_output} palabras")
print(f"   - Ratio: 1 palabra â‰ˆ {ratio_tokens} tokens")
print(f"   - Precio input: ${precio_input_por_millon} / 1M tokens")
print(f"   - Precio output: ${precio_output_por_millon} / 1M tokens")

# Pregunta 1: Tokens de input
tokens_input = palabras_input * ratio_tokens
print("\n" + "=" * 60)
print("â“ PREGUNTA 1: Â¿CuÃ¡ntos tokens de input por documento?")
print("-" * 60)
print(f"   CÃ¡lculo: {palabras_input} palabras Ã— {ratio_tokens} = {tokens_input} tokens")
print(f"   âœ… Respuesta: {tokens_input:.0f} tokens de input")

# Pregunta 2: Tokens de output
tokens_output = palabras_output * ratio_tokens
print("\n" + "=" * 60)
print("â“ PREGUNTA 2: Â¿CuÃ¡ntos tokens de output por resumen?")
print("-" * 60)
print(f"   CÃ¡lculo: {palabras_output} palabras Ã— {ratio_tokens} = {tokens_output} tokens")
print(f"   âœ… Respuesta: {tokens_output:.0f} tokens de output")

# Pregunta 3: Coste por documento
coste_input_por_doc = (tokens_input / 1_000_000) * precio_input_por_millon
coste_output_por_doc = (tokens_output / 1_000_000) * precio_output_por_millon
coste_total_por_doc = coste_input_por_doc + coste_output_por_doc

print("\n" + "=" * 60)
print("â“ PREGUNTA 3: Â¿CuÃ¡nto cuesta procesar 1 documento?")
print("-" * 60)
print(f"   Coste Input:")
print(f"      ({tokens_input} / 1,000,000) Ã— ${precio_input_por_millon} = ${coste_input_por_doc:.6f}")
print(f"   Coste Output:")
print(f"      ({tokens_output} / 1,000,000) Ã— ${precio_output_por_millon} = ${coste_output_por_doc:.6f}")
print(f"   âœ… Respuesta: ${coste_total_por_doc:.6f} por documento")
print(f"      ({coste_total_por_doc*1000:.3f} cÃ©ntimos)")

# Pregunta 4: Coste 1,000 documentos
coste_1000_docs = coste_total_por_doc * 1000
print("\n" + "=" * 60)
print("â“ PREGUNTA 4: Â¿CuÃ¡nto costarÃ­a procesar 1,000 documentos?")
print("-" * 60)
print(f"   CÃ¡lculo: ${coste_total_por_doc:.6f} Ã— 1,000 = ${coste_1000_docs:.2f}")
print(f"   âœ… Respuesta: ${coste_1000_docs:.2f}")

# Pregunta 5: Coste 10,000 documentos al mes
coste_10000_docs = coste_total_por_doc * 10000
print("\n" + "=" * 60)
print("â“ PREGUNTA 5: Â¿CuÃ¡nto costarÃ­a procesar 10,000 documentos al mes?")
print("-" * 60)
print(f"   CÃ¡lculo: ${coste_total_por_doc:.6f} Ã— 10,000 = ${coste_10000_docs:.2f}")
print(f"   âœ… Respuesta: ${coste_10000_docs:.2f} / mes")
print(f"      (â‰ˆ ${coste_10000_docs * 12:.2f} / aÃ±o)")

print("\n" + "=" * 60)
print("ðŸŽ¯ RESUMEN:")
print("=" * 60)
print(f"   1 documento: ${coste_total_por_doc:.6f}")
print(f"   1,000 documentos: ${coste_1000_docs:.2f}")
print(f"   10,000 documentos/mes: ${coste_10000_docs:.2f}")
print(f"   120,000 documentos/aÃ±o: ${coste_10000_docs * 12:.2f}")
print("=" * 60)

print("\nðŸ’¡ CONCLUSIÃ“N:")
print("   âœ… GPT-4o-mini es BARATO para esta aplicaciÃ³n")
print("   âœ… Menos de 1 cÃ©ntimo por documento")
print("   âœ… Incluso a gran volumen, el coste es razonable")
print("\n   ðŸ“Š ComparaciÃ³n:")
print(f"   - 10,000 docs/mes con GPT-4o-mini: ${coste_10000_docs:.2f}")
print(f"   - 10,000 docs/mes con GPT-4o: ${coste_10000_docs * (2.50/0.15):.2f} (16x mÃ¡s caro)")
print(f"   - 10,000 docs/mes con Claude 4 Opus: ${coste_10000_docs * (15/0.15):.2f} (100x mÃ¡s caro!)")


## 5.3 Widget Calculadora Interactiva

Ahora que sabes calcular manualmente, usa esta calculadora para estimar costes rÃ¡pidamente.

In [ ]:
# ðŸ’° Calculadora de Costes Interactiva

# Diccionario de modelos con sus precios
MODELOS = {
    "GPT-4o": {"input": 2.50, "output": 10.00},
    "GPT-4o-mini": {"input": 0.15, "output": 0.60},
    "GPT-4 Turbo": {"input": 10.00, "output": 30.00},
    "o1": {"input": 15.00, "output": 60.00},
    "o1-mini": {"input": 3.00, "output": 12.00},
    "Claude 4 Opus": {"input": 15.00, "output": 75.00},
    "Claude 4 Sonnet": {"input": 3.00, "output": 15.00},
    "Claude 4 Haiku": {"input": 0.25, "output": 1.25},
    "Gemini 3 Pro": {"input": 1.25, "output": 5.00},
    "Gemini 3 Flash": {"input": 0.075, "output": 0.30},
}

# Widgets
modelo_dropdown = widgets.Dropdown(
    options=list(MODELOS.keys()),
    value="GPT-4o-mini",
    description='Modelo:',
    layout=Layout(width='300px')
)

input_tokens_widget = widgets.IntText(
    value=1000,
    description='Input tokens:',
    layout=Layout(width='300px')
)

output_tokens_widget = widgets.IntText(
    value=500,
    description='Output tokens:',
    layout=Layout(width='300px')
)

num_llamadas_widget = widgets.IntText(
    value=1,
    description='NÂº llamadas:',
    layout=Layout(width='300px')
)

calcular_coste_button = Button(
    description='ðŸ’° Calcular Coste',
    button_style='success',
    layout=Layout(width='200px')
)

resultado_coste_output = Output()

def calcular_coste_api(b):
    with resultado_coste_output:
        resultado_coste_output.clear_output()
        
        modelo_seleccionado = modelo_dropdown.value
        input_tokens = input_tokens_widget.value
        output_tokens = output_tokens_widget.value
        num_llamadas = num_llamadas_widget.value
        
        if input_tokens <= 0 or output_tokens <= 0 or num_llamadas <= 0:
            print("âš ï¸ Todos los valores deben ser mayores que 0")
            return
        
        precios = MODELOS[modelo_seleccionado]
        precio_input = precios["input"]
        precio_output = precios["output"]
        
        # Calcular costes por llamada
        coste_input_llamada = (input_tokens / 1_000_000) * precio_input
        coste_output_llamada = (output_tokens / 1_000_000) * precio_output
        coste_total_llamada = coste_input_llamada + coste_output_llamada
        
        # Calcular costes totales
        coste_input_total = coste_input_llamada * num_llamadas
        coste_output_total = coste_output_llamada * num_llamadas
        coste_total = coste_total_llamada * num_llamadas
        
        print("=" * 60)
        print(f"ðŸ’° CALCULADORA DE COSTES - {modelo_seleccionado}")
        print("=" * 60)
        
        print(f"\nðŸ“Š CONFIGURACIÃ“N:")
        print(f"   - Modelo: {modelo_seleccionado}")
        print(f"   - Precio Input: ${precio_input} / 1M tokens")
        print(f"   - Precio Output: ${precio_output} / 1M tokens")
        print(f"   - Input por llamada: {input_tokens:,} tokens")
        print(f"   - Output por llamada: {output_tokens:,} tokens")
        print(f"   - NÃºmero de llamadas: {num_llamadas:,}")
        
        print(f"\nðŸ’µ COSTE POR LLAMADA:")
        print(f"   - Input:  ${coste_input_llamada:.6f}")
        print(f"   - Output: ${coste_output_llamada:.6f}")
        print(f"   - TOTAL:  ${coste_total_llamada:.6f} ({coste_total_llamada*1000:.3f} cÃ©ntimos)")
        
        if num_llamadas > 1:
            print(f"\nðŸ’µ COSTE TOTAL ({num_llamadas:,} llamadas):")
            print(f"   - Input:  ${coste_input_total:.2f}")
            print(f"   - Output: ${coste_output_total:.2f}")
            print(f"   - TOTAL:  ${coste_total:.2f}")
            
            # Proyecciones
            print(f"\nðŸ“ˆ PROYECCIONES:")
            print(f"   - Por dÃ­a (mismo volumen): ${coste_total:.2f}")
            print(f"   - Por mes (30 dÃ­as): ${coste_total * 30:.2f}")
            print(f"   - Por aÃ±o (365 dÃ­as): ${coste_total * 365:.2f}")
        
        # Comparar con otros modelos similares
        print(f"\nðŸ” COMPARACIÃ“N CON OTROS MODELOS:")
        print("-" * 60)
        
        comparaciones = []
        for nombre, precios_modelo in MODELOS.items():
            if nombre == modelo_seleccionado:
                continue
            coste_comparacion = (
                (input_tokens / 1_000_000 * precios_modelo["input"] * num_llamadas) +
                (output_tokens / 1_000_000 * precios_modelo["output"] * num_llamadas)
            )
            ahorro = coste_total - coste_comparacion
            porcentaje = (ahorro / coste_total * 100) if coste_total > 0 else 0
            
            comparaciones.append((nombre, coste_comparacion, ahorro, porcentaje))
        
        # Ordenar por coste (mÃ¡s barato primero)
        comparaciones.sort(key=lambda x: x[1])
        
        for i, (nombre, coste_comp, ahorro, porcentaje) in enumerate(comparaciones[:5], 1):
            if ahorro < 0:
                emoji = "ðŸ’š"
                texto = f"Ahorras ${abs(ahorro):.2f} ({abs(porcentaje):.1f}%)"
            else:
                emoji = "ðŸ“ˆ"
                texto = f"Cuesta ${abs(ahorro):.2f} mÃ¡s (+{abs(porcentaje):.1f}%)"
            
            print(f"{i}. {emoji} {nombre:20} ${coste_comp:8.2f}  |  {texto}")
        
        print("=" * 60)

calcular_coste_button.on_click(calcular_coste_api)

# Interfaz
display(VBox([
    widgets.HTML("<h3>ðŸ’° Calculadora de Costes de APIs</h3>"),
    widgets.HTML("<p style='color: #666;'>Estima cuÃ¡nto te costarÃ­a usar diferentes modelos:</p>"),
    modelo_dropdown,
    input_tokens_widget,
    output_tokens_widget,
    num_llamadas_widget,
    calcular_coste_button,
    resultado_coste_output
]))

# Ejemplos
display(widgets.HTML("""
    <div style='margin-top: 20px; padding: 15px; background-color: #e8f5e9; border-radius: 5px;'>
        <b>ðŸ’¡ Ejemplos de uso tÃ­pico:</b><br><br>
        <b>Chatbot simple:</b><br>
        â€¢ Input: 500 tokens (contexto + pregunta)<br>
        â€¢ Output: 200 tokens (respuesta corta)<br>
        â€¢ Llamadas: 1,000 / dÃ­a<br><br>
        <b>Generador de contenido:</b><br>
        â€¢ Input: 200 tokens (instrucciones)<br>
        â€¢ Output: 1,500 tokens (artÃ­culo largo)<br>
        â€¢ Llamadas: 100 / dÃ­a<br><br>
        <b>AnÃ¡lisis de documentos:</b><br>
        â€¢ Input: 5,000 tokens (documento largo)<br>
        â€¢ Output: 300 tokens (resumen)<br>
        â€¢ Llamadas: 500 / dÃ­a
    </div>
"""))


## 5.4 Comparativa Visual de Precios

Vamos a visualizar los precios de todos los modelos para que veas de un vistazo cuÃ¡l es mÃ¡s econÃ³mico.

In [ ]:
# ðŸ“Š VisualizaciÃ³n de Precios de Modelos

# Preparar datos para grÃ¡fico
modelos_lista = []
precios_input = []
precios_output = []
proveedores = []

for nombre, precios in MODELOS.items():
    modelos_lista.append(nombre)
    precios_input.append(precios["input"])
    precios_output.append(precios["output"])
    
    # Asignar proveedor
    if "GPT" in nombre or "o1" in nombre:
        proveedores.append("OpenAI")
    elif "Claude" in nombre:
        proveedores.append("Anthropic")
    elif "Gemini" in nombre:
        proveedores.append("Google")
    else:
        proveedores.append("Otro")

# Crear DataFrame
df_precios = pd.DataFrame({
    "Modelo": modelos_lista,
    "Input": precios_input,
    "Output": precios_output,
    "Proveedor": proveedores
})

# Ordenar por precio de input
df_precios = df_precios.sort_values("Input")

print("ðŸ“Š TABLA DE PRECIOS (ordenada por Input mÃ¡s barato)")
print("=" * 60)
print(df_precios.to_string(index=False))
print("=" * 60)

# GrÃ¡fico de barras interactivo con Plotly
fig = go.Figure()

# Agregar barras de Input
fig.add_trace(go.Bar(
    name='Input ($/1M tokens)',
    x=df_precios['Modelo'],
    y=df_precios['Input'],
    marker_color='lightblue',
    text=df_precios['Input'].apply(lambda x: f'${x:.2f}'),
    textposition='outside'
))

# Agregar barras de Output
fig.add_trace(go.Bar(
    name='Output ($/1M tokens)',
    x=df_precios['Modelo'],
    y=df_precios['Output'],
    marker_color='lightcoral',
    text=df_precios['Output'].apply(lambda x: f'${x:.2f}'),
    textposition='outside'
))

fig.update_layout(
    title='ðŸ’° Comparativa de Precios por Modelo ($/1M tokens)',
    xaxis_title='Modelo',
    yaxis_title='Precio ($/1M tokens)',
    barmode='group',
    height=600,
    font=dict(size=12),
    xaxis={'tickangle': -45},
    legend=dict(x=0.01, y=0.99),
    hovermode='x unified'
)

fig.show()

# GrÃ¡fico de dispersiÃ³n: Input vs Output
fig2 = px.scatter(
    df_precios,
    x='Input',
    y='Output',
    size='Output',
    color='Proveedor',
    hover_name='Modelo',
    title='ðŸ“ˆ Precio Input vs Output (tamaÃ±o = precio output)',
    labels={'Input': 'Precio Input ($/1M tokens)', 'Output': 'Precio Output ($/1M tokens)'},
    height=600,
    width=900
)

fig2.update_traces(marker=dict(line=dict(width=2, color='white')))
fig2.show()

print("\nðŸ’¡ OBSERVACIONES DEL GRÃFICO:")
print("   âœ… Gemini Flash es el MÃS BARATO para input y output")
print("   âš ï¸ Claude Opus es el MÃS CARO (pero mÃ¡xima calidad)")
print("   ðŸŽ¯ GPT-4o-mini es el mejor equilibrio precio/rendimiento")
print("   ðŸ“Š Output SIEMPRE cuesta 3-5x mÃ¡s que Input")
print("\nðŸŽ¯ RECOMENDACIÃ“N:")
print("   - Testing/Desarrollo: Gemini Flash (casi gratis)")
print("   - ProducciÃ³n alto volumen: GPT-4o-mini")
print("   - MÃ¡xima calidad sin lÃ­mite presupuesto: Claude Opus o o1")


---
# 6. âš¡ OPTIMIZA TUS TOKENS (Ahorra ðŸ’¸)

Ahora que sabes cÃ³mo se miden y cobran los tokens, aprenderÃ¡s **tÃ©cnicas prÃ¡cticas** para reducir costes sin sacrificar calidad.

## 6.1 Prompt Largo vs. Prompt Corto

El primer paso para ahorrar es **eliminar palabrerÃ­a innecesaria** de tus prompts.

### âŒ Prompt Largo e Ineficiente (Mala prÃ¡ctica)

```
Buenos dÃ­as estimado modelo de inteligencia artificial. Me dirijo a usted con el 
fin de solicitarle que, por favor, si es tan amable y tiene la disposiciÃ³n de 
hacerlo, me proporcione una explicaciÃ³n detallada, completa y exhaustiva sobre 
el concepto de lo que se conoce como machine learning, que tambiÃ©n es conocido 
en espaÃ±ol como aprendizaje automÃ¡tico. Me gustarÃ­a que la explicaciÃ³n sea lo 
mÃ¡s clara posible, de manera que pueda entenderla sin problemas. Muchas gracias 
de antemano por su atenciÃ³n y colaboraciÃ³n.
```

**ðŸ“Š AnÃ¡lisis:**
- Palabras: 98
- Tokens estimados: ~130 tokens
- Palabras Ãºtiles: ~5
- Eficiencia: âŒ 5%

---

### âœ… Prompt Corto y Eficiente (Buena prÃ¡ctica)

```
Explica machine learning de forma clara y concisa.
```

**ðŸ“Š AnÃ¡lisis:**
- Palabras: 7
- Tokens estimados: ~10 tokens
- Palabras Ãºtiles: 7
- Eficiencia: âœ… 100%
- **Ahorro: 92% de tokens** (13x mÃ¡s eficiente)

---

### ðŸŽ¯ Principios de OptimizaciÃ³n

| Principio | âŒ Evitar | âœ… Hacer |
|-----------|----------|----------|
| **CortesÃ­a excesiva** | "Por favor, si es tan amable..." | "Explica X" |
| **Redundancia** | "Detallada, completa y exhaustiva" | "Detallada" |
| **Contexto obvio** | "Buenos dÃ­as estimado modelo de IA..." | (Omitir, es obvio) |
| **Frases de relleno** | "Me gustarÃ­a que...", "si es posible..." | Imperativo directo |

---

### ðŸ’¡ Reglas de Oro para Prompts Eficientes

1. **SÃ© directo:** Usa imperativos ("Explica", "Resume", "Lista")
2. **Elimina cortesÃ­as:** El modelo no necesita "por favor" ni "gracias"
3. **Sin presentaciones:** No digas "Hola" ni "Soy X"
4. **Una idea por palabra:** No uses sinÃ³nimos encadenados
5. **Formato estructurado:** Usa listas, bullets, JSON cuando sea posible

---

### ðŸ“Š Comparativa Real de Tokens

Vamos a contar tokens reales de ambos prompts:

In [ ]:
# ðŸ“Š ComparaciÃ³n: Prompt Largo vs Corto

if TIKTOKEN_AVAILABLE:
    encoding = tiktoken.get_encoding("cl100k_base")
    
    # Prompts a comparar
    prompt_largo = """Buenos dÃ­as estimado modelo de inteligencia artificial. Me dirijo a usted con el fin de solicitarle que, por favor, si es tan amable y tiene la disposiciÃ³n de hacerlo, me proporcione una explicaciÃ³n detallada, completa y exhaustiva sobre el concepto de lo que se conoce como machine learning, que tambiÃ©n es conocido en espaÃ±ol como aprendizaje automÃ¡tico. Me gustarÃ­a que la explicaciÃ³n sea lo mÃ¡s clara posible, de manera que pueda entenderla sin problemas. Muchas gracias de antemano por su atenciÃ³n y colaboraciÃ³n."""
    
    prompt_corto = """Explica machine learning de forma clara y concisa."""
    
    # Tokenizar
    tokens_largo = encoding.encode(prompt_largo)
    tokens_corto = encoding.encode(prompt_corto)
    
    num_tokens_largo = len(tokens_largo)
    num_tokens_corto = len(tokens_corto)
    
    # Calcular ahorro
    tokens_ahorrados = num_tokens_largo - num_tokens_corto
    porcentaje_ahorro = (tokens_ahorrados / num_tokens_largo) * 100
    
    print("=" * 60)
    print("ðŸ“Š COMPARATIVA: PROMPT LARGO VS CORTO")
    print("=" * 60)
    
    print("\nâŒ PROMPT LARGO:")
    print(f"   Texto: '{prompt_largo[:80]}...'")
    print(f"   Caracteres: {len(prompt_largo)}")
    print(f"   Palabras: {len(prompt_largo.split())}")
    print(f"   TOKENS: {num_tokens_largo}")
    
    print("\nâœ… PROMPT CORTO:")
    print(f"   Texto: '{prompt_corto}'")
    print(f"   Caracteres: {len(prompt_corto)}")
    print(f"   Palabras: {len(prompt_corto.split())}")
    print(f"   TOKENS: {num_tokens_corto}")
    
    print("\nðŸ’° AHORRO:")
    print(f"   Tokens ahorrados: {tokens_ahorrados} ({porcentaje_ahorro:.1f}%)")
    print(f"   Factor de reducciÃ³n: {num_tokens_largo / num_tokens_corto:.1f}x")
    
    # Calcular ahorro en dinero
    print("\nðŸ’µ IMPACTO ECONÃ“MICO (1,000 llamadas con GPT-4o):")
    
    # Precio GPT-4o input
    precio_input = 2.50  # por 1M tokens
    
    coste_prompt_largo = (num_tokens_largo / 1_000_000) * precio_input * 1000
    coste_prompt_corto = (num_tokens_corto / 1_000_000) * precio_input * 1000
    ahorro_dinero = coste_prompt_largo - coste_prompt_corto
    
    print(f"   Prompt largo: ${coste_prompt_largo:.2f}")
    print(f"   Prompt corto: ${coste_prompt_corto:.2f}")
    print(f"   AHORRO: ${ahorro_dinero:.2f} ({porcentaje_ahorro:.1f}%)")
    
    print("\nðŸ“ˆ PROYECCIÃ“N ANUAL:")
    llamadas_por_dia = 1000
    dias_al_ano = 365
    total_llamadas = llamadas_por_dia * dias_al_ano
    
    ahorro_anual = ahorro_dinero * dias_al_ano
    print(f"   Con {llamadas_por_dia:,} llamadas/dÃ­a:")
    print(f"   Ahorro mensual: ${ahorro_anual / 12:.2f}")
    print(f"   Ahorro anual: ${ahorro_anual:.2f}")
    
    print("\n" + "=" * 60)
    print("ðŸŽ¯ CONCLUSIÃ“N:")
    print(f"   âœ… El prompt corto es {num_tokens_largo / num_tokens_corto:.1f}x mÃ¡s eficiente")
    print(f"   âœ… Ahorras {porcentaje_ahorro:.1f}% de tokens")
    print(f"   âœ… A gran escala, el ahorro es significativo")
    print(f"   âœ… La respuesta serÃ¡ la MISMA o incluso MEJOR")
    print("=" * 60)
    
else:
    print("âš ï¸ tiktoken no disponible para comparaciÃ³n")


## 6.2 Recortar Contexto Innecesario

**Problema comÃºn:** Enviar demasiado contexto que el modelo no necesita realmente.

### âŒ MAL: Contexto Completo Innecesario

```python
# Enviar TODOS los mensajes del historial
historial = [
    {"role": "user", "content": "Hola"},
    {"role": "assistant", "content": "Â¡Hola! Â¿CÃ³mo estÃ¡s?"},
    {"role": "user", "content": "Bien, gracias"},
    {"role": "assistant", "content": "Me alegro"},
    # ... 50 mensajes mÃ¡s de saludo ...
    {"role": "user", "content": "Â¿CuÃ¡l es la capital de Francia?"},
]

# Total: 5,000+ tokens innecesarios
```

### âœ… BIEN: Solo Contexto Relevante

```python
# Enviar solo los Ãºltimos N mensajes relevantes
historial_reducido = historial[-5:]  # Ãšltimos 5 mensajes
# O mejor aÃºn, solo el mensaje actual si no necesita contexto:

historial_reducido = [
    {"role": "user", "content": "Â¿CuÃ¡l es la capital de Francia?"}
]

# Total: ~15 tokens
```

---

### ðŸŽ¯ Estrategias de ReducciÃ³n de Contexto

| Estrategia | DescripciÃ³n | Ahorro tÃ­pico |
|------------|-------------|---------------|
| **Ventana deslizante** | Solo Ãºltimos N mensajes | 70-90% |
| **Resumen del historial** | Resumir conversaciÃ³n previa | 80-95% |
| **Contexto selectivo** | Solo mensajes con info clave | 60-80% |
| **Sin contexto** | Si la pregunta es independiente | 90-99% |

---

### ðŸ’¡ CuÃ¡ndo DEBES mantener contexto:

- âœ… Referencias a mensajes anteriores ("eso", "lo anterior")
- âœ… Conversaciones multi-turno con seguimiento
- âœ… Tareas que requieren recordar preferencias

### ðŸ’¡ CuÃ¡ndo PUEDES eliminar contexto:

- âŒ Saludos y cortesÃ­as("Hola", "Gracias")
- âŒ Mensajes de mÃ¡s de 10 turnos atrÃ¡s
- âŒ Preguntas independientes sin referencias
- âŒ Confirmaciones simples ("Ok", "Entendido")

## 6.3 Limitar max_tokens en Output

**El truco mÃ¡s eficazpara controlar costes:** Limitar la longitud de las respuestas.

### ðŸŽšï¸ ParÃ¡metro max_tokens

Todas las APIs permiten especificar un lÃ­mite mÃ¡ximo de tokens en la respuesta:

```python
# Sin lÃ­mite (peligroso)
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Explica Python"}]
    # max_tokens no especificado â†’ puede generar 4,096+ tokens
)

# Con lÃ­mite (controlado)
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Explica Python"}],
    max_tokens=150  # MÃ¡ximo 150 tokens de respuesta
)
```

---

### ðŸ“Š Impacto en Costes

**Ejemplo: GPT-4o (Output = $10 / 1M tokens)**

| max_tokens | Tokens reales | Coste por llamada | Coste 1,000 llamadas |
|------------|---------------|-------------------|----------------------|
| Sin lÃ­mite | 500-2000 | $0.005 - $0.020 | $5 - $20 |
| 500 | 500 | $0.005 | $5 |
| 200 | 200 | $0.002 | $2 |
| 100 | 100 | $0.001 | $1 |

**Ahorro potencial: 80-90%** en costes de output.

---

### ðŸŽ¯ Valores Recomendados de max_tokens

| Caso de Uso | max_tokens | Ejemplo |
|-------------|------------|---------|
| **Respuesta SÃ­/No** | 10-20 | "Â¿Es Python un lenguaje de programaciÃ³n?" â†’ "SÃ­" |
| **Respuesta corta** | 50-100 | "Â¿QuÃ© es Python?" â†’ "Un lenguaje de programaciÃ³n..." |
| **Respuesta normal** | 150-300 | "Explica Python" â†’ PÃ¡rrafo completo |
| **Respuesta larga** | 500-1000 | "Tutorial completo de Python" â†’ ArtÃ­culo |
| **GeneraciÃ³n de cÃ³digo** | 300-500 | "Crea una funciÃ³n" â†’ FunciÃ³n con comentarios |

---

### âš ï¸ Cuidado: No cortar demasiado

```python
# âŒ MAL: max_tokens demasiado bajo
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Explica cÃ³mo funciona internet"}],
    max_tokens=20  # â† Demasiado corto, respuesta incompleta
)
# Resultado: "Internet es una red global que conecta computadoras mediante..."
# (Se corta a mitad de frase â† MAL UX)

# âœ… BIEN: max_tokens adecuado
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Explica cÃ³mo funciona internet en 2 frases"}],
    max_tokens=100  # â† Suficiente para 2 frases completas
)
```

---

### ðŸ’¡ Estrategia Combinada: Prompt + max_tokens

```python
# âœ… Ã“PTIMO: Combinar instrucciones claras con max_tokens

# En el prompt, especifica la longitud deseada
prompt = "Explica Python en MÃXIMO 50 palabras"

# Y ademÃ¡s limita max_tokens como red de seguridad
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=80  # ~50 palabras Ã— 1.3 = 65 tokens + margen
)
```

**Resultado:** Control total sobre longitud + ahorro garantizado.

## 6.4 Ejercicio: Optimiza Este Prompt

### ðŸŽ¯ EJERCICIO FINAL

A continuaciÃ³n tienes un prompt **MUY INEFICIENTE**. Tu tarea es optimizarlo sin perder funcionalidad.

---

### âŒ PROMPT ORIGINAL (Ineficiente)

```
Hola, mi nombre es Juan y soy desarrollador de software. Trabajo en una empresa 
de tecnologÃ­a desde hace 5 aÃ±os. Ãšltimamente he estado muy interesado en el tema 
de la inteligencia artificial y el machine learning. Me gustarÃ­a aprender mÃ¡s 
sobre este fascinante campo.

Por esa razÃ³n, me dirijo a ti, que eres un modelo de inteligencia artificial 
muy avanzado y poderoso, con la esperanza de que puedas ayudarme en mi objetivo 
de aprendizaje.

Concretamente, lo que necesito es que me expliques, si es tan amable y tiene 
tiempo para ello, los siguientes conceptos fundamentales del machine learning:

1. Primero, me gustarÃ­a entender quÃ© es exactamente el aprendizaje supervisado 
   y en quÃ© se diferencia del no supervisado.

2. En segundo lugar, necesito saber cuÃ¡les son los algoritmos mÃ¡s populares y 
   utilizados en la industria actualmente.

3. Y por Ãºltimo, pero no menos importante, me interesarÃ­a conocer algunas 
   aplicaciones prÃ¡cticas reales del machine learning en el mundo empresarial.

Te agradecerÃ­a mucho si pudieras responder a estas tres preguntas de manera 
clara, concisa pero a la vez completa. No necesito explicaciones demasiado 
tÃ©cnicas, ya que soy principiante en este tema.

Muchas gracias de antemano por tu tiempo y ayuda. Espero tu respuesta con mucho 
interÃ©s.

Saludos cordiales,
Juan
```

---

### ðŸ“Š AnÃ¡lisis del Prompt Original

- **Palabras**: ~300
- **Tokens estimados**: ~420 tokens
- **InformaciÃ³n Ãºtil**: 3 preguntas simples
- **Eficiencia**: âŒ ~10%

---

### ðŸ’­ TU TURNO

**Pregunta:** Â¿CÃ³mo optimizarÃ­as este prompt?

**Objetivos:**
1. Reducir a MENOS DE 50 tokens
2. Mantener las 3 preguntas
3. No perder claridad

**ðŸ’¡ Pista:** Elimina todo lo innecesario (presentaciones, cortesÃ­as, contexto personal...)

---

### â¬ DesplÃ¡zate para ver la soluciÃ³n...

<br><br><br><br><br>

In [ ]:
# ðŸŽ¯ SOLUCIÃ“N DEL EJERCICIO

if TIKTOKEN_AVAILABLE:
    encoding = tiktoken.get_encoding("cl100k_base")
    
    # Prompt original (ineficiente)
    prompt_original = """Hola, mi nombre es Juan y soy desarrollador de software. Trabajo en una empresa de tecnologÃ­a desde hace 5 aÃ±os. Ãšltimamente he estado muy interesado en el tema de la inteligencia artificial y el machine learning. Me gustarÃ­a aprender mÃ¡s sobre este fascinante campo.

Por esa razÃ³n, me dirijo a ti, que eres un modelo de inteligencia artificial muy avanzado y poderoso, con la esperanza de que puedas ayudarme en mi objetivo de aprendizaje.

Concretamente, lo que necesito es que me expliques, si es tan amable y tiene tiempo para ello, los siguientes conceptos fundamentales del machine learning:

1. Primero, me gustarÃ­a entender quÃ© es exactamente el aprendizaje supervisado y en quÃ© se diferencia del no supervisado.

2. En segundo lugar, necesito saber cuÃ¡les son los algoritmos mÃ¡s populares y utilizados en la industria actualmente.

3. Y por Ãºltimo, pero no menos importante, me interesarÃ­a conocer algunas aplicaciones prÃ¡cticas reales del machine learning en el mundo empresarial.

Te agradecerÃ­a mucho si pudieras responder a estas tres preguntas de manera clara, concisa pero a la vez completa. No necesito explicaciones demasiado tÃ©cnicas, ya que soy principiante en este tema.

Muchas gracias de antemano por tu tiempo y ayuda. Espero tu respuesta con mucho interÃ©s.

Saludos cordiales,
Juan"""
    
    # Prompt optimizado
    prompt_optimizado = """Explica para principiantes:
1. Aprendizaje supervisado vs no supervisado
2. Algoritmos ML mÃ¡s populares
3. Aplicaciones reales en empresas"""
    
    # Tokenizar
    tokens_original = encoding.encode(prompt_original)
    tokens_optimizado = encoding.encode(prompt_optimizado)
    
    num_original = len(tokens_original)
    num_optimizado = len(tokens_optimizado)
    
    tokens_ahorrados = num_original - num_optimizado
    porcentaje_ahorro = (tokens_ahorrados / num_original) * 100
    factor_reduccion = num_original / num_optimizado
    
    print("=" * 60)
    print("âœ… SOLUCIÃ“N: PROMPT OPTIMIZADO")
    print("=" * 60)
    
    print("\nðŸ“ PROMPT OPTIMIZADO:")
    print("-" * 60)
    print(prompt_optimizado)
    print("-" * 60)
    
    print("\nðŸ“Š COMPARATIVA:")
    print("=" * 60)
    print(f"{'MÃ©trica':<25} {'Original':>15} {'Optimizado':>15}")
    print("-" * 60)
    print(f"{'Tokens':<25} {num_original:>15} {num_optimizado:>15}")
    print(f"{'Palabras':<25} {len(prompt_original.split()):>15} {len(prompt_optimizado.split()):>15}")
    print(f"{'Caracteres':<25} {len(prompt_original):>15} {len(prompt_optimizado):>15}")
    print("=" * 60)
    
    print(f"\nðŸ’° AHORRO:")
    print(f"   Tokens ahorrados: {tokens_ahorrados} ({porcentaje_ahorro:.1f}%)")
    print(f"   Factor de reducciÃ³n: {factor_reduccion:.1f}x")
    
    # Impacto econÃ³mico
    print(f"\nðŸ’µ IMPACTO ECONÃ“MICO (10,000 llamadas/mes con GPT-4o):")
    
    precio_input = 2.50  # por 1M tokens
    llamadas = 10000
    
    coste_original = (num_original / 1_000_000) * precio_input * llamadas
    coste_optimizado = (num_optimizado / 1_000_000) * precio_input * llamadas
    ahorro_mensual = coste_original - coste_optimizado
    ahorro_anual = ahorro_mensual * 12
    
    print(f"   Original: ${coste_original:.2f}/mes")
    print(f"   Optimizado: ${coste_optimizado:.2f}/mes")
    print(f"   AHORRO: ${ahorro_mensual:.2f}/mes (${ahorro_anual:.2f}/aÃ±o)")
    
    print("\nðŸŽ¯ TÃ‰CNICAS APLICADAS:")
    print("   âœ… Eliminada presentaciÃ³n personal (innecesaria)")
    print("   âœ… Eliminadas cortesÃ­as y frases de relleno")
    print("   âœ… Eliminado contexto personal irrelevante")
    print("   âœ… Convertido a formato de lista (mÃ¡s eficiente)")
    print("   âœ… Directa peticiÃ³n: 'Explica para principiantes'")
    print("   âœ… Mantenidas las 3 preguntas esenciales")
    
    print("\nðŸ’¡ RESULTADO:")
    print(f"   âœ… {factor_reduccion:.1f}x mÃ¡s eficiente")
    print(f"   âœ… Misma funcionalidad")
    print(f"   âœ… Respuesta serÃ¡ igual o MEJOR (prompt mÃ¡s claro)")
    print(f"   âœ… Ahorro de ${ahorro_anual:.2f}/aÃ±o a escala")
    
    print("=" * 60)
    
else:
    print("âš ï¸ tiktoken no disponible")
    print("\nâœ… SOLUCIÃ“N (sin contar tokens):")
    print("""
Explica para principiantes:
1. Aprendizaje supervisado vs no supervisado
2. Algoritmos ML mÃ¡s populares
3. Aplicaciones reales en empresas
    """)
    print("\nTÃ‰CNICAS: Eliminada toda presentaciÃ³n, cortesÃ­as y contexto innecesario.")
    print("RESULTADO: ~30 tokens vs ~420 originales (14x mÃ¡s eficiente)")


---
# ðŸŽ“ RESUMEN Y CONCLUSIONES

Â¡Felicidades! Has completado el Notebook 2 sobre Embeddings, Tokens y Costes.

## ðŸ“š QuÃ© Has Aprendido

### 1. ðŸ§¬ EMBEDDINGS
- âœ… Los embeddings son **vectores numÃ©ricos que capturan el significado** de textos
- âœ… Palabras/frases similares tienen vectores cercanos en el espacio
- âœ… Se usan para: bÃºsqueda semÃ¡ntica, recomendaciones, clasificaciÃ³n
- âœ… Hay **modelos API** (Google, OpenAI) y **locales** (Sentence-Transformers)
- âœ… **Google text-embedding-004** es el mejor para empezar (gratis + calidad)

**ðŸŽ¯ Regla prÃ¡ctica:** Para bÃºsquedas por significado (no palabras exactas), usa embeddings + similitud coseno.

---

### 2. ðŸŽ« TOKENS
- âœ… Los tokens son **pedazos de texto** (ni palabras ni caracteres)
- âœ… **1 token â‰ˆ 0.75 palabras** en inglÃ©s, **1 token â‰ˆ 1.3 palabras** en espaÃ±ol
- âœ… EspaÃ±ol usa **30-60% MÃS tokens** que inglÃ©s (acentos, palabras largas)
- âœ… Las APIs **cobran por token**, no por palabra ni carÃ¡cter
- âœ… Hay **3 tipos**: Input (lo que envÃ­as), Output (lo que recibes), Reasoning (interno) 
- âœ… Usa **tiktoken** para contar tokens reales antes de enviar a API

**ðŸŽ¯ Regla prÃ¡ctica:** Siempre estima tokens ANTES de hacer llamadas masivas:
```python
tokens_estimados = palabras Ã— 1.3  # espaÃ±ol
coste = (tokens / 1_000_000) Ã— precio_modelo
```

---

### 3. ðŸ’° COSTES
- âœ… Los modelos cobran por **millÃ³n de tokens** ($X.XX / 1M tokens)
- âœ… **Output cuesta 3-5x MÃS** que input
- âœ… **Gemini Flash** es el mÃ¡s barato ($0.075 input, $0.30 output)
- âœ… **GPT-4o-mini** es el mejor equilibrio (barato + buena calidad)
- âœ… **Claude Opus** y **o1** son los mÃ¡s caros (mÃ¡xima calidad)

**ðŸŽ¯ Comparativa rÃ¡pida (Input / 1M tokens):**
- ðŸ†“ Gemini 2.5 Flash Lite: **GRATIS**
- ðŸ’š Gemini 3 Flash: **$0.075**
- ðŸ’› GPT-4o-mini: **$0.15**
- ðŸ§¡ GPT-4o: **$2.50**
- â¤ï¸ o1 / Claude Opus: **$15.00**

---

### 4. âš¡ OPTIMIZACIÃ“N
- âœ… **Elimina palabrerÃ­a:** Prompts directos son 10-20x mÃ¡s eficientes
- âœ… **Recorta contexto:** Solo envÃ­a los Ãºltimos N mensajes necesarios
- âœ… **Usa max_tokens:** Limita la longitud de respuestas
- âœ… **Elige el modelo adecuado:** No uses GPT-4 si GPT-4o-mini es suficiente

**ðŸŽ¯ TÃ©cnicas de ahorro:**
| TÃ©cnica | Ahorro tÃ­pico | CuÃ¡ndo usarla |
|---------|---------------|---------------|
| Prompts cortos | 70-90% | Siempre |
| Limitar contexto | 60-80% | Chatbots con historial |
| max_tokens | 50-70% | Respuestas predecibles |
| Modelo mÃ¡s barato | 80-95% | Tareas simples |

---

## ðŸŽ¯ Consejos Finales del Profesor

### Para Estudiantes / Proyectos PequeÃ±os:
1. **Usa Gemini 3 Flash** (gratis + excelente calidad)
2. **O modelos locales** si valoras privacidad (Sentence-Transformers)
3. **Aprende a contar tokens** antes de empezar proyectos grandes
4. **Optimiza desde el dÃ­a 1** (mal hÃ¡bito = dinero desperdiciado)

### Para Empresas / ProducciÃ³n:
1. **Empieza con GPT-4o-mini o Gemini Flash** (precio/rendimiento)
2. **Monitoriza costes diarios** (sorpresas desagradables = problema comÃºn)
3. **Implementa cachÃ©** si envÃ­as mismo contexto repetidamente
4. **Usa modelos caros solo cuando necesites** (o1 para lÃ³gica, Claude Opus para creatividad)
5. **Considera modelos locales** para datos sensibles (HIPAA, GDPR)

---

## ðŸ“ˆ PrÃ³ximos Pasos

### âœ… Lo que YA sabes hacer:
1. Generar embeddings (API y local)
2. Calcular similitudes semÃ¡nticas
3. Construir bÃºsquedas semÃ¡nticas bÃ¡sicas
4. Contar tokens de cualquier texto
5. Estimar costes de APIs
6. Optimizar prompts para ahorrar dinero

### ðŸš€ Lo que aprenderÃ¡s en Notebook 3:
- **Prompting Avanzado:** Chain-of-Thought, Few-Shot, Prompts estructurados
- **RAG (Retrieval Augmented Generation):** Chatbots con documentos propios
- **Memory y Context Management:** Conversaciones largas eficientes
- **Function Calling:** LLMs que ejecutan cÃ³digo/APIs

---

## ðŸ“ Ejercicios Recomendados (Hazlos por tu cuenta)

1. **Buscador de documentos:** Crea un buscador semÃ¡ntico con 100 documentos propios (PDFs, artÃ­culos)
2. **Calculadora de presupuesto:** Estima el coste mensual de tu idea de aplicaciÃ³n con  LLMs
3. **Comparar embeddings:** Compara Google VS Sentence-Transformers en TU caso de uso
4. **Optimizar prompts reales:** Toma 10 prompts de tu trabajo y optimÃ­zalos
5. **Monitor de tokens:** Crea un script que cuente tokens ANTES de enviar a API

---

## ðŸŽ‰ Â¡Felicidades!

Has completado uno de los notebooks mÃ¡s importantes del curso. Estos conceptos (embeddings, tokens, costes) son **FUNDAMENTALES** para ser un profesional competente en IA Generativa.

**ðŸ’¡ Recuerda:**
- Los embeddings son el "idioma secreto" de las IAs
- Los tokens son la "moneda" con la que se mide todo
- Optimizar costes no es opcional, es obligatorio a escala
- Siempre hay un modelo mÃ¡s barato que hace el 80% del trabajo

---

## ðŸ“š Recursos Adicionales

### DocumentaciÃ³n Oficial:
- **Google Gemini Embeddings:** https://ai.google.dev/tutorials/embeddings_quickstart
- **OpenAI Embeddings:** https://platform.openai.com/docs/guides/embeddings
- **Sentence-Transformers:** https://www.sbert.net/
- **Tiktoken:** https://github.com/openai/tiktoken

### Benchmarks y Comparativas:
- **MTEB Leaderboard:** https://huggingface.co/spaces/mteb/leaderboard
- **LLM Price Comparison:** https://www.artificialanalysis.ai/models

### Herramientas Ãštiles:
- **Tokenizador OpenAI:** https://platform.openai.com/tokenizer
- **Embeddings Visualizer:** https://projector.tensorflow.org/

---

## ðŸ™ CrÃ©ditos y Agradecimientos

Este notebook fue creado con â¤ï¸ para ayudarte a dominar conceptos cruciales de IA Generativa.

**Si tienes dudas:**
- ðŸ“§ Contacta al profesor
- ðŸ’¬ Ãšnete a la comunidad del curso
- ðŸ› Reporta bugs o mejoras sugeridas

---

### â­ PrÃ³ximo Notebook: 03_Prompting_Avanzado_RAG.ipynb

Nos vemos allÃ­. **Â¡Sigue aprendiendo!** ðŸš€